In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kgan31/doha-simple-context/dohas_nlp_ready_simple_lang.csv
/kaggle/input/datasets/kgan31/charan-seperated-doha/doha_charan_seperated.csv


In [2]:
!pip install -q transformers datasets accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 10.4 MB/s eta 0:00:00a 0:00:01


In [7]:
import torch
import pandas as pd
import os
from datasets import Dataset
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)

# -------------------------------------------------------------------
# 0. SETUP & AUTHENTICATION
# -------------------------------------------------------------------
try:
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Successfully logged into Hugging Face Hub!")
except Exception as e:
    print("Running without HF Token.")
    HF_TOKEN = None

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "KGan31/Doha-Gen" # Your Stage 1 Model

print("Loading Model and Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("google/byt5-small")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)
model.config.tie_word_embeddings = False 

# -------------------------------------------------------------------
# 1. PREPARE CONTROL-TOKEN DATASET
# -------------------------------------------------------------------
print("Preparing dataset with Control Tokens...")
csv_path = '/kaggle/input/datasets/kgan31/doha-simple-context/dohas_nlp_ready_simple_lang.csv' # Ensure this is your correct Kaggle path
df = pd.read_csv(csv_path)
df = df.dropna(subset=['Theme', 'Context', 'Doha'])

# NOVELTY: Injecting the structural rule directly into the prompt
df['input_text'] = "विषय: " + df['Theme'] + " | संदर्भ: " + df['Context'] + " | नियम: 13-11-13-11 मात्रा"
df['target_text'] = df['Doha'].astype(str).str.replace('\n', ' \\ ').str.replace('\r', '')

dataset = Dataset.from_pandas(df[['input_text', 'target_text']])
dataset = dataset.train_test_split(test_size=0.05)

# Tokenize function
MAX_LENGTH = 512
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"], 
        max_length=MAX_LENGTH, 
        truncation=True
    )
    labels = tokenizer(
        text_target=examples["target_text"], 
        max_length=MAX_LENGTH, 
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing data...")
tokenized_datasets = dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=dataset["train"].column_names
)

# -------------------------------------------------------------------
# 2. CONFIGURE TRAINING
# -------------------------------------------------------------------
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/byt5-doha-stage2",
    eval_strategy="epoch",
    learning_rate=5e-5,            # Standard fine-tuning rate
    per_device_train_batch_size=4, # Fits easily on Kaggle
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    num_train_epochs=5,            # 5 epochs for the model to memorize the 13-11 rule
    predict_with_generate=True,
    fp16=True,                     # Mixed precision for speed
    logging_steps=50,
    report_to="none",
    save_total_limit=2,
    push_to_hub=True if HF_TOKEN else False,
    hub_model_id="KGan31/Doha-Gen-Stage2" if HF_TOKEN else None,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

# -------------------------------------------------------------------
# 3. START TRAINING
# -------------------------------------------------------------------
print("Starting Stage 2: Control-Token Fine-Tuning...")
trainer.train()

trainer.save_model("/kaggle/working/byt5-doha-stage2-final")

if HF_TOKEN:
    print("Pushing final Stage 2 model to Hugging Face Hub...")
    trainer.push_to_hub(commit_message="Completed Stage 2: Control-Token SFT for Prosody")
    print("Upload complete!")

Successfully logged into Hugging Face Hub!
Loading Model and Tokenizer...


Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Preparing dataset with Control Tokens...
Tokenizing data...


Map:   0%|          | 0/7494 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Upload complete!


In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -------------------------------------------------------------------
# 1. SETUP & LOAD MODEL
# -------------------------------------------------------------------
# Using the repo ID where you pushed the Stage 2 model
MODEL_ID = "KGan31/Doha-Gen-Stage2" 
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading tokenizer and model from: {MODEL_ID} (Device: {device})")

# Load tokenizer and model directly from your Hugging Face repo
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID).to(device)

# -------------------------------------------------------------------
# 2. DEFINE TEST CASES
# -------------------------------------------------------------------
# Add some sample themes and contexts to evaluate how well the model
# adheres to the prompt structure and the 13-11-13-11 rule.
test_cases = [
    {
        "theme": "प्रेम", 
        "context": "सच्चा प्रेम निस्वार्थ होता है और ईश्वर की प्राप्ति का मार्ग है।"
    },
    {
        "theme": "विद्या", 
        "context": "ज्ञान से बड़ा कोई धन नहीं है, इसे जितना बांटो उतना बढ़ता है।"
    },
    {
        "theme": "समय", 
        "context": "बीता हुआ समय कभी लौटकर नहीं आता, इसलिए वर्तमान का सदुपयोग करो।"
    },
    {
        "theme": "प्रकृति", 
        "context": "प्रकृति हमारी माता के समान है, इसका संरक्षण हमारा कर्तव्य है।"
    },
    {
        "theme": "मित्रता", 
        "context": "विपत्ति के समय ही सच्चे मित्र की पहचान होती है।"
    }
]

# -------------------------------------------------------------------
# 3. GENERATION FUNCTION
# -------------------------------------------------------------------
def generate_doha(theme, context, model, tokenizer, device):
    # Construct the prompt exactly as it was during fine-tuning
    prompt = f"विषय: {theme} | संदर्भ: {context} | नियम: 13-11-13-11 मात्रा"
    
    inputs = tokenizer(
        prompt, 
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=512
    ).to(device)

    # Generate output
    # Adjust parameters like num_beams, temperature, or top_p to tweak creativity vs strictness
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=400,          
            do_sample=True,          
            top_p=0.85,              # Slightly more restricted vocabulary
            temperature=0.7,         # Lower temperature to regain coherence
            repetition_penalty=1.05, # ONLY 1.05! Never go high with byte-level models
            num_beams=1              
        )
    # Decode the generated tokens back to text
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Format the output for readability (replacing the literal ' \ ' back to newlines)
    formatted_doha = generated_text.replace(' \\ ', '\n')
    
    return prompt, formatted_doha

# -------------------------------------------------------------------
# 4. RUN HUMAN EVALUATION
# -------------------------------------------------------------------
print("\n" + "="*50)
print("GENERATING DOHAS FOR HUMAN EVALUATION")
print("="*50 + "\n")

for i, test in enumerate(test_cases, 1):
    prompt, doha = generate_doha(test["theme"], test["context"], model, tokenizer, device)
    
    print(f"Test Case {i}:")
    print(f"Prompt: {prompt}")
    print("-" * 30)
    print(f"Generated Doha:\n{doha}")
    print("-" * 30)
    print("Evaluation Notes (Prosody/Meaning):")
    print("13-11 Rule met? [ ] Yes  [ ] No")
    print("Context match?  [ ] Yes  [ ] No\n")
    print("="*50 + "\n")

Loading tokenizer and model from: KGan31/Doha-Gen-Stage2 (Device: cuda)


Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



GENERATING DOHAS FOR HUMAN EVALUATION

Test Case 1:
Prompt: विषय: प्रेम | संदर्भ: सच्चा प्रेम निस्वार्थ होता है और ईश्वर की प्राप्ति का मार्ग है। | नियम: 13-11-13-11 मात्रा
------------------------------
Generated Doha:
स्वामी सुरी होती है मन के साथ है अभिराम।
देखि हरि होत है तन प्राप्ति में जो भाग ॥
------------------------------
Evaluation Notes (Prosody/Meaning):
13-11 Rule met? [ ] Yes  [ ] No
Context match?  [ ] Yes  [ ] No


Test Case 2:
Prompt: विषय: विद्या | संदर्भ: ज्ञान से बड़ा कोई धन नहीं है, इसे जितना बांटो उतना बढ़ता है। | नियम: 13-11-13-11 मात्रा
------------------------------
Generated Doha:
ज्ञान को कोई पानिक को, सारा कोई नहीं है।
कोई मेरा कोई धन है, इसे जितना बांटो उतना बढ़ता है ॥
------------------------------
Evaluation Notes (Prosody/Meaning):
13-11 Rule met? [ ] Yes  [ ] No
Context match?  [ ] Yes  [ ] No


Test Case 3:
Prompt: विषय: समय | संदर्भ: बीता हुआ समय कभी लौटकर नहीं आता, इसलिए वर्तमान का सदुपयोग करो। | नियम: 13-11-13-11 मात्रा
----------------------------

In [3]:
# ─── stage2_train_kaggle.py ──────────────────────────────────────────────────
"""
Stage 2: Theme-conditioned Doha generation with ByT5 +
         span-level differentiable matra loss.

Fixes applied (see review doc):
  A. Leading-byte gradient limitation — continuation byte positions now also
     receive a small gradient via P(valid_continuation | pos), weighted by
     CONT_GRAD_WEIGHT=0.1. This steers the model toward valid UTF-8 sequences
     without overpowering the primary matra signal at the leading byte.
  B. Segment-bound teacher-forcing / inference loops — seg_bounds are now
     pre-computed inside DohaDataset.__init__ (once, at load time) so they
     are always aligned with the tokenizer's output. generate_doha() now
     also uses repetition_penalty=1.3 to prevent '||' separator loops.
  C. Halant conjunct rule (laghu→guru) — syllabify() now applies a second
     pass that upgrades a laghu syllable to guru when it is immediately
     followed by a conjunct cluster (halant-internal syllable or zero-matra
     halant unit), matching classical Chhand Shastra prosody rules.
  D. Character-by-character syllabify — _build_char_matra_table() now
     receives the full raw charan string and calls syllabify() on it once,
     then aligns the resulting context-aware matra values back to individual
     byte-token positions. Conjunct and anusvara context rules are respected.
  E. Vocabulary size safety — DohaByT5.__init__ reads the ACTUAL embedding
     rows from the loaded checkpoint and uses min(vocab_size, actual_V) for
     all runtime masking, avoiding index-out-of-bounds if the checkpoint was
     resized. _build_non_cont_mask() clamps CONT_HIGH to actual_V-1.
"""

import ast, os, json, math, unicodedata
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import (
    T5ForConditionalGeneration,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from huggingface_hub import login, HfApi


# ════════════════════════════════════════════════════════════════════════════
# 0.  SETUP
# ════════════════════════════════════════════════════════════════════════════

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("✓ Logged into Hugging Face Hub")
except Exception as e:
    print(f"No HF Token: {e}")
    HF_TOKEN = None

STAGE1_MODEL_ID      = "KGan31/Doha-Gen"
STAGE2_HUB_ID        = "KGan31/Doha-Gen-Stage2_Matra_loss_integrated"
CSV_PATH             = '/kaggle/input/datasets/kgan31/charan-seperated-doha/doha_charan_seperated.csv'
OUTPUT_DIR           = '/kaggle/working/stage2_checkpoints'
FINAL_MODEL_DIR      = '/kaggle/working/stage2_final'

NUM_EPOCHS           = 20
BATCH_SIZE           = 4
GRAD_ACCUM_STEPS     = 4
LR                   = 3e-5
WARMUP_RATIO         = 0.1
LAMBDA_MAX           = 0.5
LAMBDA_WARMUP_EPOCHS = 5
VAL_SPLIT            = 0.1
MAX_SRC              = 256
MAX_TGT              = 200
SEED                 = 42

# ByT5 byte-token offset: token_id = byte_value + 3
BYT5_BYTE_OFFSET = 3
# Safe ceiling — actual vocab size is read from the model in DohaByT5.__init__
VOCAB_SIZE       = 384

# Continuation byte token range: UTF-8 bytes 0x80–0xBF → token_ids 131–194
CONT_LOW  = 0x80 + BYT5_BYTE_OFFSET   # 131
CONT_HIGH = 0xBF + BYT5_BYTE_OFFSET   # 194
SPECIAL   = {0, 1, 2}                 # pad, eos, unk

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
torch.manual_seed(SEED)


# ════════════════════════════════════════════════════════════════════════════
# 1.  MATRA UTILITIES
# ════════════════════════════════════════════════════════════════════════════

HALANT, ANUSVARA, VISARGA, NUKTA = '\u094D', '\u0902', '\u0903', '\u093C'
CONSONANTS = set(range(0x0915, 0x093A)) | set(range(0x0958, 0x0960))
IND_VOWELS = set(range(0x0905, 0x0915))
DEP_VOWELS = set(range(0x093E, 0x094D)) | {0x094E, 0x094F}
GURU_DEP   = {'\u093E', '\u0940', '\u0942', '\u0947',
              '\u0948', '\u094B', '\u094C', '\u094F'}
GURU_IND   = {'\u0906', '\u0908', '\u090A', '\u090F',
              '\u0910', '\u0913', '\u0914', '\u0960'}
DEVANAGARI_DIGIT_MATRAS = {chr(0x0966 + i): v for i, v in
                            enumerate([2, 1, 2, 1, 2, 2, 2, 2, 2, 2])}
ARABIC_DIGIT_MATRAS = {str(d): 2 for d in range(10)}


def syllabify(text: str) -> list:
    """
    Segment a Hindi string into syllables and return their matra weights.

    FIX C — Halant/conjunct laghu→guru rule:
        Classical Chhand Shastra: a laghu (1-matra) syllable immediately
        preceding a conjunct cluster (samyukta akshara, halant-joined
        consonants) is counted as guru (2 matras) because the compound
        pronunciation is heavy.

        Implementation: after the main parsing pass, a second pass upgrades
        any laghu syllable at index k to guru if syllable k+1 either:
          (a) has matras==0 (trailing virama / pure-halant unit), or
          (b) its text contains HALANT not at the final position (internal
              conjunct joiner, e.g. 'त्' in 'त्य').
    """
    text = unicodedata.normalize('NFC', text)
    chars, n, syllables, i = list(text), len(text), [], 0

    while i < n:
        ch, cp = chars[i], ord(chars[i])

        if cp in range(0x0966, 0x0970):
            syllables.append({'text': ch,
                               'matras': DEVANAGARI_DIGIT_MATRAS.get(ch, 2)})
            i += 1

        elif ch.isdigit():
            syllables.append({'text': ch,
                               'matras': ARABIC_DIGIT_MATRAS.get(ch, 2)})
            i += 1

        elif cp in IND_VOWELS:
            unit = ch; i += 1
            while i < n and ord(chars[i]) in {0x0902, 0x0903, 0x0901}:
                unit += chars[i]; i += 1
            guru = ch in GURU_IND or ANUSVARA in unit or VISARGA in unit
            syllables.append({'text': unit, 'matras': 2 if guru else 1})

        elif cp in CONSONANTS:
            unit = ch; i += 1
            if i < n and chars[i] == NUKTA:
                unit += chars[i]; i += 1

            # Consume internal conjunct consonants (C + halant + C ...)
            while (i + 1 < n and chars[i] == HALANT
                   and ord(chars[i + 1]) in CONSONANTS):
                unit += chars[i] + chars[i + 1]; i += 2
                if i < n and chars[i] == NUKTA:
                    unit += chars[i]; i += 1

            # Trailing halant (virama) — pure consonant, zero matra
            if i < n and chars[i] == HALANT:
                unit += chars[i]; i += 1
                syllables.append({'text': unit, 'matras': 0})
                continue

            # Dependent vowel
            dep = ''
            if i < n and ord(chars[i]) in DEP_VOWELS:
                dep = chars[i]; i += 1

            # Modifiers (anusvara / visarga / chandrabindu)
            mod = ''
            while i < n and ord(chars[i]) in {0x0902, 0x0903, 0x0901}:
                mod += chars[i]; i += 1

            unit += dep + mod
            guru = (dep in GURU_DEP) or bool(
                mod and (ANUSVARA in mod or VISARGA in mod))
            syllables.append({'text': unit, 'matras': 2 if guru else 1})

        else:
            i += 1   # punctuation, spaces, separators — skip silently

    # ── FIX C: second pass — laghu before conjunct cluster becomes guru ──
    for k in range(len(syllables) - 1):
        if syllables[k]['matras'] != 1:
            continue
        nxt = syllables[k + 1]['text']
        # Internal halant = halant appears before the last character
        has_internal_halant = HALANT in nxt[:-1]
        next_is_zero        = (syllables[k + 1]['matras'] == 0)
        if has_internal_halant or next_is_zero:
            syllables[k] = dict(syllables[k], matras=2)

    return syllables


def total_matras(text: str) -> int:
    return sum(s['matras'] for s in syllabify(text))


# ════════════════════════════════════════════════════════════════════════════
# 2.  SPAN-LEVEL MATRA UTILITIES
# ════════════════════════════════════════════════════════════════════════════

def _is_continuation(token_id: int) -> bool:
    """True for ByT5 tokens that represent UTF-8 continuation bytes (0x80–0xBF)."""
    return CONT_LOW <= token_id <= CONT_HIGH


def _span_to_char(token_ids: list) -> str | None:
    """
    Decode a list of ByT5 token IDs representing one UTF-8 character.
    Returns the character string, or None on invalid byte sequences.
    """
    try:
        raw = bytes(tid - BYT5_BYTE_OFFSET for tid in token_ids)
        return raw.decode('utf-8')
    except (UnicodeDecodeError, ValueError):
        return None


def _extract_char_spans(token_ids: list) -> list:
    """
    Return list of (start, end) index pairs (exclusive end) for each
    character span in the token sequence.

    A new character starts at every token that is:
      - NOT a continuation byte (token_id outside [131, 194])
      - NOT a special token (0=pad, 1=eos, 2=unk)

    Examples:
      'मा' → [227,167,177, 227,164,190] → [(0,3), (3,6)]
      'म ' → [227,167,177, 35]          → [(0,3), (3,4)]
    """
    spans = []
    i, n = 0, len(token_ids)
    while i < n:
        tid = token_ids[i]
        if tid in SPECIAL or _is_continuation(tid):
            i += 1
            continue
        j = i + 1
        while j < n and _is_continuation(token_ids[j]):
            j += 1
        spans.append((i, j))
        i = j
    return spans


def _build_char_matra_table(token_ids: list, charan_text: str | None = None) -> dict:
    """
    FIX D (Revised): Properly distributes syllable weights to character positions.
    """
    if charan_text is None:
        # Fallback to decoding if text isn't provided
        spans = _extract_char_spans(token_ids)
        charan_text = "".join([_span_to_char(token_ids[s:e]) or "" for s, e in spans])

    # 1. Get context-aware syllables (e.g., ['सत्', 'य'])
    syllables = syllabify(charan_text)
    
    # 2. Map every Unicode character in the text to its correct weight
    # We want to assign the weight to the 'vowel' part of the syllable
    full_text_char_weights = []
    for syl in syllables:
        syl_chars = list(unicodedata.normalize('NFC', syl['text']))
        weight = float(syl['matras'])
        
        # Logic: Find the last character that isn't a Halant to carry the weight
        # In 'सत्', 'स' gets the weight, 'त्' gets 0.
        # In 'य', 'य' gets the weight.
        main_carrier_idx = -1
        for i in range(len(syl_chars)-1, -1, -1):
            if syl_chars[i] != HALANT:
                main_carrier_idx = i
                break
        
        for i in range(len(syl_chars)):
            full_text_char_weights.append(weight if i == main_carrier_idx else 0.0)

    # 3. Align these weights to the actual token spans
    table = {}
    spans = _extract_char_spans(token_ids)
    
    # Safety: align based on character count
    for idx, (start, end) in enumerate(spans):
        if idx < len(full_text_char_weights):
            table[start] = full_text_char_weights[idx]
        else:
            table[start] = 0.0
            
    return table


# ════════════════════════════════════════════════════════════════════════════
# 3.  DIFFERENTIABLE EXPECTED MATRA
# ════════════════════════════════════════════════════════════════════════════

def _build_non_cont_mask(vocab_size: int,
                          dtype: torch.dtype,
                          device: torch.device) -> torch.Tensor:
    """
    (V,) mask: 1.0 for tokens that can START a new character, 0.0 otherwise.

    FIX E: uses actual vocab_size (from embedding table) instead of the global
    constant, and clamps CONT_HIGH to vocab_size-1 so slicing never goes OOB.
    """
    mask = torch.ones(vocab_size, dtype=dtype, device=device)
    for tid in SPECIAL:
        if tid < vocab_size:
            mask[tid] = 0.0
    lo = min(CONT_LOW,  vocab_size)
    hi = min(CONT_HIGH, vocab_size - 1)
    if lo <= hi:
        mask[lo:hi + 1] = 0.0
    return mask


def _build_cont_mask(vocab_size: int,
                     dtype: torch.dtype,
                     device: torch.device) -> torch.Tensor:
    """(V,) mask: 1.0 for valid UTF-8 continuation byte tokens."""
    mask = torch.zeros(vocab_size, dtype=dtype, device=device)
    lo = min(CONT_LOW,  vocab_size)
    hi = min(CONT_HIGH, vocab_size - 1)
    if lo <= hi:
        mask[lo:hi + 1] = 1.0
    return mask


def compute_expected_matra_span(
    logits: torch.Tensor,         # (T, V) float32, grad-attached
    matra_table: dict,            # {position: matra_value}
    seg_start: int,
    seg_end: int,
    non_cont_mask: torch.Tensor,  # (V,) — tokens that can start a char
    cont_mask: torch.Tensor,      # (V,) — valid continuation byte tokens
) -> torch.Tensor:
    """
    Differentiable expected matra for one charan segment.

    FIX A — continuation-byte gradient:
        At each leading-byte position s, the primary gradient is:
            matra_val × P(valid_start_byte | position=s)   [unchanged]

        At each continuation position s+k (k=1,2,...), we add:
            γ × P(valid_continuation_byte | position=s+k)
        where γ=CONT_GRAD_WEIGHT=0.1.

        This means the model is penalised for emitting non-continuation bytes
        in the middle of a multi-byte character (which would produce garbled
        UTF-8 and make _span_to_char return None, silently dropping the matra
        contribution). The continuation gradient is small enough that it does
        not distort the matra signal.

    FIX E (via callers): non_cont_mask and cont_mask are sized to actual_V,
        so no index can go out of bounds even if V < VOCAB_SIZE.
    """
    T, V = logits.shape
    CONT_GRAD_WEIGHT = 0.1   # γ — weight for continuation-byte regularisation

    total_expected = logits.new_zeros(1).squeeze()

    # Character-start positions in this segment, sorted
    sorted_starts = sorted(p for p in matra_table if seg_start <= p < seg_end)

    for idx, pos in enumerate(sorted_starts):
        matra_val = matra_table[pos]
        if pos >= T:
            continue

        # ── Leading byte: primary matra gradient ─────────────────────────
        probs   = torch.softmax(logits[pos], dim=-1)           # (V,)
        p_start = (probs * non_cont_mask).sum()
        if matra_val != 0.0:
            total_expected = total_expected + matra_val * p_start

        # ── Continuation bytes: UTF-8 validity regularisation (Fix A) ────
        # Continuation positions run from pos+1 to (next start - 1).
        next_start = (sorted_starts[idx + 1]
                      if idx + 1 < len(sorted_starts) else seg_end)
        next_start = min(next_start, T)
        for cont_pos in range(pos + 1, next_start):
            if cont_pos >= T:
                break
            p_cont = (torch.softmax(logits[cont_pos], dim=-1) * cont_mask).sum()
            total_expected = total_expected + CONT_GRAD_WEIGHT * p_cont

    return total_expected


# ════════════════════════════════════════════════════════════════════════════
# 4.  DATASET
# ════════════════════════════════════════════════════════════════════════════

DOHA_TARGETS      = [13.0, 11.0, 13.0, 11.0]
CHARAN_SEPARATORS = [' | ', ' || ', ' | ', '']   # after c1, c2, c3, c4


def _charan_byte_bounds(charans: list, tokenizer, max_tgt: int) -> list:
    """
    Compute token-position boundaries [0, e1, e2, e3, e4] for the 4 charans.

    FIX B: called once inside DohaDataset.__init__ so seg_bounds are stable
    and always aligned with the tokenizer's deterministic encoding of each
    charan string (not recomputed per batch inside collate_fn).
    """
    bounds = [0]
    pos    = 0
    for k, charan in enumerate(charans):
        pos += len(tokenizer.encode(charan, add_special_tokens=False))
        sep  = CHARAN_SEPARATORS[k]
        if sep:
            pos += len(tokenizer.encode(sep, add_special_tokens=False))
        bounds.append(min(pos, max_tgt))
    return bounds   # 5 elements: [0, e1, e2, e3, e4]


class DohaDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer,
                 max_src: int = 256, max_tgt: int = 200):
        self.records = []
        for _, row in df.iterrows():
            try:
                charans = (ast.literal_eval(row['charans'])
                           if isinstance(row['charans'], str)
                           else row['charans'])
                matras  = (ast.literal_eval(row['matras'])
                           if isinstance(row['matras'], str)
                           else row['matras'])
            except Exception:
                charans, matras = [], []

            valid = bool(row.get('valid', False)) and len(charans) == 4

            # FIX B: pre-compute seg_bounds at load time
            seg_bounds = (
                _charan_byte_bounds(charans, tokenizer, max_tgt)
                if valid else None
            )

            self.records.append({
                'input':      f"विषय: {row['theme']} | संदर्भ: {row['context']}",
                'target':     str(row['formatted_doha']),
                'charans':    charans if valid else [],
                'matras':     matras,
                'valid':      valid,
                'seg_bounds': seg_bounds,
            })

    def __len__(self):          return len(self.records)
    def __getitem__(self, idx): return self.records[idx]


def collate_fn(batch, tokenizer, max_src: int = 256, max_tgt: int = 200):
    enc = tokenizer(
        [b['input']  for b in batch],
        padding=True, truncation=True,
        max_length=max_src, return_tensors='pt',
    )
    dec = tokenizer(
        text_target=[b['target'] for b in batch],
        padding=True, truncation=True,
        max_length=max_tgt, return_tensors='pt',
    )

    labels = dec.input_ids.clone()
    if dec.attention_mask is not None:
        labels[dec.attention_mask == 0] = -100

    return {
        'input_ids':      enc.input_ids,
        'attention_mask': enc.attention_mask,
        'labels':         labels,
        'seg_bounds':     [b['seg_bounds'] for b in batch],   # pre-computed (Fix B)
        'charans':        [b['charans']    for b in batch],   # raw strings  (Fix D)
        'gold_matras':    [b['matras']     for b in batch],
        'valid':          [b['valid']      for b in batch],
    }


# ════════════════════════════════════════════════════════════════════════════
# 5.  DIAGNOSTICS
# ════════════════════════════════════════════════════════════════════════════

def diagnose_all(df, train_loader, tokenizer, model):
    print("=" * 60)
    print("CSV DIAGNOSTICS")
    print("=" * 60)
    print(f"Shape  : {df.shape}")
    print(f"Nulls  :\n{df[['theme','context','formatted_doha','charans','matras','valid']].isnull().sum()}")
    print(f"Valid  : {df['valid'].value_counts().to_dict()}")

    print("\n" + "=" * 60)
    print("MODEL / TOKENIZER DIAGNOSTICS  (Fix E)")
    print("=" * 60)
    emb_shape = model.model.shared.weight.shape
    actual_V  = emb_shape[0]
    print(f"Embedding shape  : {emb_shape}   (rows=vocab, cols=d_model)")
    print(f"tokenizer vocab  : {tokenizer.vocab_size}")
    print(f"VOCAB_SIZE const : {VOCAB_SIZE}  (ceiling)  model.vocab_size: {model.vocab_size}")
    print(f"CONT range       : [{CONT_LOW}, {min(CONT_HIGH, actual_V - 1)}]")
    if actual_V < VOCAB_SIZE:
        print(f"  ⚠ Embedding rows ({actual_V}) < VOCAB_SIZE ({VOCAB_SIZE}) "
              f"— model.vocab_size clamped to {actual_V}")
    else:
        print(f"  ✓ Embedding rows ≥ VOCAB_SIZE constant")

    sample   = 'म'
    enc_ids  = tokenizer.encode(sample, add_special_tokens=False)
    expected = [b + BYT5_BYTE_OFFSET for b in sample.encode('utf-8')]
    match    = enc_ids == expected
    print(f"Offset check 'म': encoded={enc_ids}  expected={expected}  ✓={match}")

    print("\n" + "=" * 60)
    print("BATCH DIAGNOSTICS")
    print("=" * 60)
    batch      = next(iter(train_loader))
    labels     = batch['labels']
    non_masked = labels != -100
    visible    = labels[non_masked]

    print(f"input_ids shape : {batch['input_ids'].shape}")
    print(f"labels shape    : {labels.shape}")
    print(f"Non-masked      : {non_masked.sum().item()} / {labels.numel()} "
          f"({100*non_masked.float().mean():.1f}%)")
    print(f"Label id range  : {visible.min().item()} – {visible.max().item()}")
    print(f"Embedding rows  : {actual_V}")
    if visible.max().item() >= actual_V:
        print("  ✗ LABEL ID EXCEEDS EMBEDDING TABLE — will cause nan!")
    else:
        print("  ✓ All label ids within embedding range")
    print(f"\nDecoded sample  : {tokenizer.decode(visible[:60])}")

    model.eval()
    with torch.no_grad():
        raw_out = model.model(
            input_ids      = batch['input_ids'].to(device),
            attention_mask = batch['attention_mask'].to(device),
            labels         = batch['labels'].to(device),
        )
    loss_val = raw_out.loss.item()
    print(f"\nFloat32 CE (raw model, no AMP) : {loss_val:.6f}")
    print(f"Is finite                      : {math.isfinite(loss_val)}")
    if not math.isfinite(loss_val):
        print("  ✗ Still nan without AMP → deeper issue")
    else:
        print("  ✓ Finite without AMP → AMP overflow was the only problem")

    # ── Fix C: conjunct / halant spot-checks ─────────────────────────────
    print("\n" + "=" * 60)
    print("SYLLABIFY / HALANT DIAGNOSTICS  (Fix C)")
    print("=" * 60)
    halant_cases = [
        ('सत्य',  [2, 1],    "स before त्य conjunct → guru"),
        ('धर्म',  [2, 1],    "ध before र्म conjunct → guru"),
        ('कर्म',  [2, 1],    "क before र्म → guru"),
        ('राम',   [1, 1, 1], "no conjunct, all laghu"),
        ('मात्र', [1, 2, 1], "ा makes मा guru; त्र conjunct makes त्र zero"),
        ('सत्',   [2],       "trailing halant — zero unit triggers upgrade of स"),
    ]
    all_ok = True
    for text, expected_matras, note in halant_cases:
        syls   = syllabify(text)
        got    = [s['matras'] for s in syls]
        ok     = (got == expected_matras)
        status = '✓' if ok else '✗'
        if not ok:
            all_ok = False
        print(f"  {status} '{text}': got {got}  expected {expected_matras}  ({note})")
    if not all_ok:
        print("  ⚠ Some cases differ — review syllabify() conjunct rules")

    # ── Fix D: span matra table spot-checks ──────────────────────────────
    print("\n" + "=" * 60)
    print("SPAN MATRA TABLE DIAGNOSTICS  (Fix D)")
    print("=" * 60)
    for text in ['मा', 'सत्य', 'राम', 'कर्म']:
        tids  = tokenizer.encode(text, add_special_tokens=False)
        table = _build_char_matra_table(tids, charan_text=text)
        spans = _extract_char_spans(tids)
        print(f"  '{text}' → token_ids={tids}")
        for s, e in spans:
            ch = _span_to_char(tids[s:e])
            print(f"    span [{s}:{e}] = '{ch}'  matra={table.get(s, '?')}")

    print("=" * 60 + "\n")
    model.train()


# ════════════════════════════════════════════════════════════════════════════
# 6.  MODEL
# ════════════════════════════════════════════════════════════════════════════

class DohaByT5(nn.Module):
    def __init__(self, model_name: str, lambda_matra: float = 0.0,
                 vocab_size: int = VOCAB_SIZE):
        super().__init__()
        self.model = T5ForConditionalGeneration.from_pretrained(model_name)

        # FIX E: clamp to actual embedding rows in case checkpoint was resized
        actual_V = self.model.shared.weight.shape[0]
        if vocab_size > actual_V:
            print(f"  ⚠ vocab_size={vocab_size} > actual embedding rows={actual_V}."
                  f" Using {actual_V}.")
        self.vocab_size   = min(vocab_size, actual_V)
        self.lambda_matra = lambda_matra

        self.register_buffer(
            'targets',
            torch.tensor(DOHA_TARGETS, dtype=torch.float32)
        )

        # Pre-build masks on CPU; moved to correct device/dtype on first call.
        self._non_cont_mask_cpu = _build_non_cont_mask(
            self.vocab_size, torch.float32, torch.device('cpu'))
        self._cont_mask_cpu     = _build_cont_mask(
            self.vocab_size, torch.float32, torch.device('cpu'))

    def _masks(self, dtype, dev):
        return (self._non_cont_mask_cpu.to(dtype=dtype, device=dev),
                self._cont_mask_cpu.to(dtype=dtype, device=dev))

    def compute_matra_loss(self,
                           logits: torch.Tensor,    # (B, T, V) float32
                           labels: torch.Tensor,    # (B, T)
                           seg_bounds: list,
                           charans_batch: list) -> torch.Tensor:
        """
        Span-level differentiable matra loss — all five fixes integrated.

        Per sample:
          1. Recover ground-truth token ids (replace -100 → 0).
          2. Per charan segment, call _build_char_matra_table with the raw
             charan string (Fix D) for context-aware syllabify (Fix C).
          3. Remap table keys from segment-local to full-sequence positions.
          4. compute_expected_matra_span with continuation gradients (Fix A)
             and bounds-safe masks (Fix E).
          5. SmoothL1(M̂_k, target_k) per segment, averaged.
        """
        B, T, V      = logits.shape
        tgts         = self.targets.to(dtype=logits.dtype, device=logits.device)
        non_cont, cont = self._masks(logits.dtype, logits.device)

        seg_loss_list = []

        for b in range(B):
            bounds  = seg_bounds[b]
            charans = charans_batch[b]   # list of 4 raw charan strings
            if bounds is None or len(bounds) < 5:
                continue

            label_ids = labels[b].tolist()
            clean_ids = [tid if tid != -100 else 0 for tid in label_ids]

            sample_losses = []
            for k in range(4):
                s, e = bounds[k], bounds[k + 1]
                if s >= e or e > T:
                    continue

                # FIX D: pass full charan string for context-aware matra lookup
                charan_text = charans[k] if k < len(charans) else None

                # Build table on the segment slice; remap keys to full positions
                local_table = _build_char_matra_table(
                    clean_ids[s:e], charan_text=charan_text)
                matra_table = {s + lp: mv for lp, mv in local_table.items()}

                M_hat_k = compute_expected_matra_span(
                    logits[b],
                    matra_table,
                    seg_start     = s,
                    seg_end       = e,
                    non_cont_mask = non_cont,
                    cont_mask     = cont,
                )

                sample_losses.append(
                    F.smooth_l1_loss(M_hat_k, tgts[k], reduction='sum'))

            if sample_losses:
                seg_loss_list.append(torch.stack(sample_losses).mean())

        if seg_loss_list:
            return torch.stack(seg_loss_list).mean()
        return (logits * 0.0).sum()   # differentiable zero

    def forward(self, input_ids, attention_mask, labels,
                seg_bounds, charans, gold_matras, **kwargs) -> dict:
        # float32 always — ByT5 overflows float16 early in training
        with torch.amp.autocast('cuda', enabled=False):
            outputs = self.model(
                input_ids      = input_ids,
                attention_mask = attention_mask,
                labels         = labels,
            )

        logits = outputs.logits.float()   # (B, T, V) attached, float32

        # CE with label smoothing
        active = labels != -100
        if active.any():
            L_ce = F.cross_entropy(
                logits[active],
                labels[active],
                label_smoothing=0.1,
                reduction='mean',
            )
        else:
            L_ce = logits.new_zeros(1).squeeze()

        if not torch.isfinite(L_ce):
            print(f"  ⚠ CE non-finite ({L_ce.item()}), zeroing.")
            L_ce = logits.new_zeros(1).squeeze()

        # Span-level matra loss
        if self.lambda_matra > 0.0:
            L_matra = self.compute_matra_loss(
                logits, labels, seg_bounds, charans)
            L_total = L_ce + self.lambda_matra * L_matra
        else:
            L_matra = logits.new_zeros(1).squeeze()
            L_total = L_ce

        return {
            'loss':       L_total,
            'ce_loss':    L_ce.detach(),
            'matra_loss': L_matra.detach(),
        }


# ════════════════════════════════════════════════════════════════════════════
# 7.  TRAINING
# ════════════════════════════════════════════════════════════════════════════

def train():
    os.makedirs(OUTPUT_DIR,      exist_ok=True)
    os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

    print(f"Loading tokenizer and model from '{STAGE1_MODEL_ID}'...")
    tokenizer = AutoTokenizer.from_pretrained(STAGE1_MODEL_ID)
    model     = DohaByT5(STAGE1_MODEL_ID, lambda_matra=0.0,
                         vocab_size=VOCAB_SIZE).to(device)
    print(f"✓ Model loaded — embedding: {model.model.shared.weight.shape} "
          f"| effective vocab_size: {model.vocab_size}")

    df      = pd.read_csv(CSV_PATH)
    full_ds = DohaDataset(df, tokenizer, MAX_SRC, MAX_TGT)
    val_n   = int(len(full_ds) * VAL_SPLIT)
    train_ds, val_ds = random_split(
        full_ds, [len(full_ds) - val_n, val_n],
        generator=torch.Generator().manual_seed(SEED),
    )
    print(f"Train: {len(train_ds)} | Val: {val_n}")

    _col         = lambda b: collate_fn(b, tokenizer, MAX_SRC, MAX_TGT)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=_col, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              collate_fn=_col, num_workers=2, pin_memory=True)

    diagnose_all(df, train_loader, tokenizer, model)

    optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps  = (len(train_loader) // GRAD_ACCUM_STEPS) * NUM_EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, warmup_steps, total_steps)

    scaler = torch.amp.GradScaler('cuda')

    best_val_loss = float('inf')
    history       = []

    for epoch in range(1, NUM_EPOCHS + 1):

        # Lambda schedule: 0 → LAMBDA_MAX over LAMBDA_WARMUP_EPOCHS
        if epoch == 1:
            model.lambda_matra = 0.0
        elif epoch <= LAMBDA_WARMUP_EPOCHS:
            model.lambda_matra = LAMBDA_MAX * (epoch - 1) / (LAMBDA_WARMUP_EPOCHS - 1)
        else:
            model.lambda_matra = LAMBDA_MAX

        # ── Train ────────────────────────────────────────────────────────
        model.train()
        tr_ce = tr_matra = tr_total = 0.0
        nan_count = 0
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader):
            out = model(
                input_ids      = batch['input_ids'].to(device),
                attention_mask = batch['attention_mask'].to(device),
                labels         = batch['labels'].to(device),
                seg_bounds     = batch['seg_bounds'],
                charans        = batch['charans'],
                gold_matras    = batch['gold_matras'],
            )
            loss = out['loss'] / GRAD_ACCUM_STEPS

            if not torch.isfinite(out['loss']):
                nan_count += 1
                optimizer.zero_grad()
                continue

            scaler.scale(loss).backward()

            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    model.parameters(), 1.0)

                if not math.isfinite(grad_norm):
                    optimizer.zero_grad()
                    scaler.update()
                    continue

                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            tr_ce    += out['ce_loss'].item()
            tr_matra += out['matra_loss'].item()
            tr_total += out['loss'].item()

            if (step + 1) % 50 == 0:
                print(f"  Ep {epoch:02d} | Step {step+1:04d}/{len(train_loader)} "
                      f"| CE {tr_ce/(step+1):.4f} "
                      f"| Matra {tr_matra/(step+1):.4f} "
                      f"| λ={model.lambda_matra:.3f}")

        n      = len(train_loader)
        avg_tr = {'ce': tr_ce/n, 'matra': tr_matra/n, 'total': tr_total/n}
        if nan_count:
            print(f"  ⚠ {nan_count} nan steps skipped this epoch")

        # ── Validate ─────────────────────────────────────────────────────
        model.eval()
        va_ce = va_matra = va_total = 0.0
        with torch.no_grad():
            for batch in val_loader:
                out = model(
                    input_ids      = batch['input_ids'].to(device),
                    attention_mask = batch['attention_mask'].to(device),
                    labels         = batch['labels'].to(device),
                    seg_bounds     = batch['seg_bounds'],
                    charans        = batch['charans'],
                    gold_matras    = batch['gold_matras'],
                )
                va_ce    += out['ce_loss'].item()
                va_matra += out['matra_loss'].item()
                va_total += out['loss'].item()

        m      = len(val_loader)
        avg_va = {'ce': va_ce/m, 'matra': va_matra/m, 'total': va_total/m}

        print(f"\nEpoch {epoch:02d} | λ={model.lambda_matra:.3f}")
        print(f"  Train → CE:{avg_tr['ce']:.4f} Matra:{avg_tr['matra']:.4f} "
              f"Total:{avg_tr['total']:.4f}")
        print(f"  Val   → CE:{avg_va['ce']:.4f} Matra:{avg_va['matra']:.4f} "
              f"Total:{avg_va['total']:.4f}\n")

        history.append({'epoch': epoch, 'train': avg_tr,
                        'val': avg_va, 'lambda': model.lambda_matra})

        if avg_va['total'] < best_val_loss:
            best_val_loss = avg_va['total']
            ckpt          = os.path.join(OUTPUT_DIR, 'best_model')
            model.model.save_pretrained(ckpt)
            tokenizer.save_pretrained(ckpt)
            torch.save({'lambda_matra': model.lambda_matra, 'epoch': epoch},
                       os.path.join(ckpt, 'meta.pt'))
            print(f"  ✓ Saved best (val_loss={best_val_loss:.4f})")

        with open(os.path.join(OUTPUT_DIR, 'history.json'), 'w',
                  encoding='utf-8') as f:
            json.dump(history, f, indent=2, ensure_ascii=False)

    # ── Final save & push ─────────────────────────────────────────────────
    model.model.save_pretrained(FINAL_MODEL_DIR)
    tokenizer.save_pretrained(FINAL_MODEL_DIR)
    torch.save({'lambda_matra': model.lambda_matra, 'epoch': NUM_EPOCHS},
               os.path.join(FINAL_MODEL_DIR, 'meta.pt'))

    if HF_TOKEN:
        api = HfApi()
        api.create_repo(repo_id=STAGE2_HUB_ID, exist_ok=True, token=HF_TOKEN)
        api.upload_folder(
            folder_path    = os.path.join(OUTPUT_DIR, 'best_model'),
            repo_id        = STAGE2_HUB_ID,
            commit_message = f"Stage 2 — val_loss={best_val_loss:.4f}, span matra loss",
            token          = HF_TOKEN,
        )
        print(f"✓ Pushed → https://huggingface.co/{STAGE2_HUB_ID}")

    return model, tokenizer


# ════════════════════════════════════════════════════════════════════════════
# 8.  INFERENCE
# ════════════════════════════════════════════════════════════════════════════

def _parse_formatted(text: str) -> list:
    text  = text.replace('||', '|')
    parts = [p.strip() for p in text.split('|')]
    return [p for p in parts if p]


def generate_doha(model, tokenizer, theme: str, context: str = '',
                  num_beams: int = 10, max_new_tokens: int = 150,
                  tolerance: int = 1) -> dict:
    """
    FIX B (inference): repetition_penalty=1.3 prevents the model from looping
    on separator tokens ('|', '||') when the matra loss is active and the
    model overshoots a charan boundary.
    """
    TARGETS = [13, 11, 13, 11]
    inputs  = tokenizer(
        f"विषय: {theme} | संदर्भ: {context}",
        return_tensors='pt').to(device)

    model.eval()
    with torch.no_grad():
        out = model.model.generate(
            **inputs,
            num_beams            = num_beams,
            num_return_sequences = num_beams,
            max_new_tokens       = max_new_tokens,
            early_stopping       = True,
            repetition_penalty   = 1.3,   # Fix B: prevents '||' separator loops
        )

    candidates  = tokenizer.batch_decode(out, skip_special_tokens=True)
    best, best_score, best_matras = None, float('inf'), []

    for cand in candidates:
        charans = _parse_formatted(cand)
        if len(charans) != 4:
            continue
        m     = [total_matras(c) for c in charans]
        score = sum(abs(m[k] - TARGETS[k]) for k in range(4))
        if score < best_score:
            best_score, best, best_matras = score, cand, m

    return {
        'doha':           best or candidates[0],
        'matras':         best_matras,
        'matra_error':    best_score,
        'valid':          best_score <= tolerance * 4,
        'all_candidates': candidates,
    }


# ════════════════════════════════════════════════════════════════════════════
# 9.  ENTRY POINT
# ════════════════════════════════════════════════════════════════════════════

if __name__ == '__main__':
    model, tokenizer = train()

    result = generate_doha(
        model, tokenizer,
        theme   = 'शृंगार',
        context = 'नायिका की सुंदरता का वर्णन',
    )
    print("\nGenerated Doha:")
    print(result['doha'])
    print(f"Matras      : {result['matras']}")
    print(f"Matra error : {result['matra_error']}")
    print(f"Valid       : {result['valid']}")

✓ Logged into Hugging Face Hub
Device: cuda
Loading tokenizer and model from 'KGan31/Doha-Gen'...


Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✓ Model loaded — embedding: torch.Size([384, 1472]) | effective vocab_size: 384
Train: 7101 | Val: 788
CSV DIAGNOSTICS
Shape  : (7889, 9)
Nulls  :
theme             0
context           0
formatted_doha    0
charans           0
matras            0
valid             0
dtype: int64
Valid  : {True: 7146, False: 743}

MODEL / TOKENIZER DIAGNOSTICS  (Fix E)
Embedding shape  : torch.Size([384, 1472])   (rows=vocab, cols=d_model)
tokenizer vocab  : 256
VOCAB_SIZE const : 384  (ceiling)  model.vocab_size: 384
CONT range       : [131, 194]
  ✓ Embedding rows ≥ VOCAB_SIZE constant
Offset check 'म': encoded=[227, 167, 177]  expected=[227, 167, 177]  ✓=True

BATCH DIAGNOSTICS
input_ids shape : torch.Size([4, 100])
labels shape    : torch.Size([4, 200])
Non-masked      : 781 / 800 (97.6%)
Label id range  : 1 – 227
Embedding rows  : 384
  ✓ All label ids within embedding range

Decoded sample  : निष्ठाएँ घायल हुईं, | पा

Float32 CE (raw model, no AMP) : 1.672343
Is finite                      : True


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Saved best (val_loss=1.6892)
  Ep 02 | Step 0050/1776 | CE 1.7815 | Matra 1.2297 | λ=0.125
  Ep 02 | Step 0100/1776 | CE 1.7880 | Matra 1.0711 | λ=0.125
  Ep 02 | Step 0150/1776 | CE 1.7921 | Matra 0.9117 | λ=0.125
  Ep 02 | Step 0200/1776 | CE 1.7906 | Matra 0.7973 | λ=0.125
  Ep 02 | Step 0250/1776 | CE 1.7884 | Matra 0.7232 | λ=0.125
  Ep 02 | Step 0300/1776 | CE 1.7853 | Matra 0.6713 | λ=0.125
  Ep 02 | Step 0350/1776 | CE 1.7828 | Matra 0.6239 | λ=0.125
  Ep 02 | Step 0400/1776 | CE 1.7805 | Matra 0.5897 | λ=0.125
  Ep 02 | Step 0450/1776 | CE 1.7782 | Matra 0.5659 | λ=0.125
  Ep 02 | Step 0500/1776 | CE 1.7757 | Matra 0.5497 | λ=0.125
  Ep 02 | Step 0550/1776 | CE 1.7744 | Matra 0.5356 | λ=0.125
  Ep 02 | Step 0600/1776 | CE 1.7722 | Matra 0.5168 | λ=0.125
  Ep 02 | Step 0650/1776 | CE 1.7708 | Matra 0.5060 | λ=0.125
  Ep 02 | Step 0700/1776 | CE 1.7692 | Matra 0.4962 | λ=0.125
  Ep 02 | Step 0750/1776 | CE 1.7677 | Matra 0.4812 | λ=0.125
  Ep 02 | Step 0800/1776 | CE 1.7663 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✓ Pushed → https://huggingface.co/KGan31/Doha-Gen-Stage2_Matra_loss_integrated

Generated Doha:
नायिका की सुंदरता में, | नायिका की सुंदरता || नायिक
Matras      : []
Matra error : inf
Valid       : False


In [5]:
def diagnose_csv(df):
    print("=" * 60)
    print("CSV DIAGNOSTICS")
    print("=" * 60)
    print(f"Columns     : {df.columns.tolist()}")
    print(f"Shape       : {df.shape}")
    print(f"\nNull counts:")
    print(df[['theme','context','formatted_doha','charans','matras','valid']].isnull().sum())
    
    print(f"\nSample formatted_doha:")
    for i, v in enumerate(df['formatted_doha'].head(3)):
        print(f"  [{i}] {repr(v)}")
    
    print(f"\nSample input that will be fed to encoder:")
    for i, row in df.head(3).iterrows():
        inp = f"विषय: {row['theme']} | संदर्भ: {row['context']}"
        print(f"  [{i}] {repr(inp[:80])}")
    
    print(f"\nValid distribution: {df['valid'].value_counts().to_dict()}")
    
    # Check for multiline formatted_doha breaking CSV parsing
    multiline = df['formatted_doha'].str.contains('\n', na=False).sum()
    print(f"\nformatted_doha rows with \\n: {multiline}")
    
    # Check charans parses correctly
    sample_charans = df['charans'].iloc[0]
    print(f"\nSample charans raw : {repr(sample_charans)}")
    try:
        import ast
        parsed = ast.literal_eval(sample_charans)
        print(f"Sample charans parsed: {parsed}")
    except Exception as e:
        print(f"charans parse ERROR: {e}")


df = pd.read_csv(CSV_PATH)
diagnose_csv(df)
print(df.columns.tolist())
print(df['formatted_doha'].head(3))
print(df['context'].head(3))

CSV DIAGNOSTICS
Columns     : ['author', 'theme', 'context', 'original_doha', 'formatted_doha', 'charans', 'matras', 'valid', 'error']
Shape       : (7889, 9)

Null counts:
theme             0
context           0
formatted_doha    0
charans           0
matras            0
valid             0
dtype: int64

Sample formatted_doha:
  [0] 'मोर पच्छ जो सिर चढ़ै | बारन तें अधिकाय || सहस चखन लखि धनि कचन | परे मान छिन पाय'
  [1] 'बेनी बधि इक ठौर ह्वै | अहि सम राखत ठौर || बिथुरि चँवरि से कच करत | मन बिथोरि धरि चौंर'
  [2] 'जे हरि रह त्रिलोक मों | कालीनाथ कहाइ || ते तुव बेनी के डसे | सब जग हँसे बनाइ'

Sample input that will be fed to encoder:
  [0] 'विषय: शृंगार | संदर्भ: मोरपंखी बाल'
  [1] 'विषय: शृंगार | संदर्भ: नागिन सी चोटी'
  [2] 'विषय: शृंगार | संदर्भ: चोटी का जादू'

Valid distribution: {True: 7146, False: 743}

formatted_doha rows with \n: 5

Sample charans raw : "['मोर पच्छ जो सिर चढ़ै', 'बारन तें अधिकाय', 'सहस चखन लखि धनि कचन', 'परे मान छिन पाय']"
Sample charans parsed: ['मोर पच्छ जो सिर

In [8]:
# ─── stage2_train_kaggle.py ──────────────────────────────────────────────────
"""
Stage 2: Theme-conditioned Doha generation with ByT5 +
         span-level differentiable matra loss.

Fixes applied (see review doc):
  A. Leading-byte gradient — continuation byte positions receive a small
     gradient via P(valid_continuation | pos), weighted by CONT_GRAD_WEIGHT=0.1.
  B. Segment-bound alignment — seg_bounds pre-computed in DohaDataset.__init__.
     generate_doha() uses repetition_penalty=1.3 to prevent '||' loops.
  C. Halant conjunct rule (laghu→guru) — syllabify() second pass upgrades
     a laghu syllable to guru when followed by a conjunct cluster.
  D. Context-aware matra table — _build_char_matra_table() syllabifies the
     full charan string and aligns results back to byte-token positions using
     a decoded-string walk rather than a flat per-codepoint expansion.
     Matra weight is assigned to the FIRST codepoint of each syllable.
  E. Vocabulary size safety — DohaByT5.__init__ clamps to actual embedding rows.

Additional corrections vs prior version:
  - Diagnostic expected values for 'राम', 'मात्र', 'सत्' were wrong;
    corrected to match actual Hindi prosody rules.
  - _build_char_matra_table "last non-halant" carrier logic replaced with
    "first codepoint of syllable" — dependent vowels (ā, ī, …) and halant
    are modifiers, not independent syllable heads.
  - Syllable-to-span alignment now walks the decoded string character by
    character and matches each syllable's codepoints to token spans in order,
    handling the case where the standalone halant byte (्, U+094D) has its
    own span in the token sequence but belongs to the preceding syllable.
"""

import ast, os, json, math, unicodedata
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import (
    T5ForConditionalGeneration,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from huggingface_hub import login, HfApi


# ════════════════════════════════════════════════════════════════════════════
# 0.  SETUP
# ════════════════════════════════════════════════════════════════════════════

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("✓ Logged into Hugging Face Hub")
except Exception as e:
    print(f"No HF Token: {e}")
    HF_TOKEN = None

STAGE1_MODEL_ID      = "KGan31/Doha-Gen"
STAGE2_HUB_ID        = "KGan31/Doha-Gen-Stage2_Matra_loss_integrated"
CSV_PATH             = '/kaggle/input/datasets/kgan31/charan-seperated-doha/doha_charan_seperated.csv'
OUTPUT_DIR           = '/kaggle/working/stage2_checkpoints'
FINAL_MODEL_DIR      = '/kaggle/working/stage2_final'

NUM_EPOCHS           = 20
BATCH_SIZE           = 4
GRAD_ACCUM_STEPS     = 4
LR                   = 3e-5
WARMUP_RATIO         = 0.1
LAMBDA_MAX           = 0.5
LAMBDA_WARMUP_EPOCHS = 5
VAL_SPLIT            = 0.1
MAX_SRC              = 256
MAX_TGT              = 200
SEED                 = 42

BYT5_BYTE_OFFSET = 3
VOCAB_SIZE       = 384   # safe ceiling; actual size read from model

CONT_LOW  = 0x80 + BYT5_BYTE_OFFSET   # 131
CONT_HIGH = 0xBF + BYT5_BYTE_OFFSET   # 194
SPECIAL   = {0, 1, 2}                 # pad, eos, unk

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
torch.manual_seed(SEED)


# ════════════════════════════════════════════════════════════════════════════
# 1.  MATRA UTILITIES
# ════════════════════════════════════════════════════════════════════════════

HALANT, ANUSVARA, VISARGA, NUKTA = '\u094D', '\u0902', '\u0903', '\u093C'
CONSONANTS = set(range(0x0915, 0x093A)) | set(range(0x0958, 0x0960))
IND_VOWELS = set(range(0x0905, 0x0915))
DEP_VOWELS = set(range(0x093E, 0x094D)) | {0x094E, 0x094F}
GURU_DEP   = {'\u093E', '\u0940', '\u0942', '\u0947',
              '\u0948', '\u094B', '\u094C', '\u094F'}
GURU_IND   = {'\u0906', '\u0908', '\u090A', '\u090F',
              '\u0910', '\u0913', '\u0914', '\u0960'}
DEVANAGARI_DIGIT_MATRAS = {chr(0x0966 + i): v for i, v in
                            enumerate([2, 1, 2, 1, 2, 2, 2, 2, 2, 2])}
ARABIC_DIGIT_MATRAS = {str(d): 2 for d in range(10)}


def syllabify(text: str) -> list:
    """
    Segment a Hindi string into syllables with matra weights.

    Returns a list of dicts: [{'text': str, 'matras': int}, ...]

    Fix C — laghu→guru before conjunct cluster:
        After the main parse, a second pass upgrades any laghu (matra=1)
        syllable to guru (matra=2) when immediately followed by a syllable
        that begins a conjunct cluster. A conjunct start is detected by:
          (a) syllable[k+1].matras == 0  (trailing halant / virama unit), or
          (b) syllable[k+1].text contains HALANT before its last character
              (internal conjunct joiner, e.g. '्' in 'त्य').
    """
    text = unicodedata.normalize('NFC', text)
    chars, n, syllables, i = list(text), len(text), [], 0

    while i < n:
        ch, cp = chars[i], ord(chars[i])

        if cp in range(0x0966, 0x0970):
            syllables.append({'text': ch,
                               'matras': DEVANAGARI_DIGIT_MATRAS.get(ch, 2)})
            i += 1

        elif ch.isdigit():
            syllables.append({'text': ch,
                               'matras': ARABIC_DIGIT_MATRAS.get(ch, 2)})
            i += 1

        elif cp in IND_VOWELS:
            unit = ch; i += 1
            while i < n and ord(chars[i]) in {0x0902, 0x0903, 0x0901}:
                unit += chars[i]; i += 1
            guru = ch in GURU_IND or ANUSVARA in unit or VISARGA in unit
            syllables.append({'text': unit, 'matras': 2 if guru else 1})

        elif cp in CONSONANTS:
            unit = ch; i += 1
            if i < n and chars[i] == NUKTA:
                unit += chars[i]; i += 1

            # Consume internal conjunct: C + halant + C (+ optional nukta) ...
            while (i + 1 < n and chars[i] == HALANT
                   and ord(chars[i + 1]) in CONSONANTS):
                unit += chars[i] + chars[i + 1]; i += 2
                if i < n and chars[i] == NUKTA:
                    unit += chars[i]; i += 1

            # Trailing halant — pure consonant skeleton, zero matra
            if i < n and chars[i] == HALANT:
                unit += chars[i]; i += 1
                syllables.append({'text': unit, 'matras': 0})
                continue

            # Dependent vowel
            dep = ''
            if i < n and ord(chars[i]) in DEP_VOWELS:
                dep = chars[i]; i += 1

            # Modifiers: anusvara / visarga / chandrabindu
            mod = ''
            while i < n and ord(chars[i]) in {0x0902, 0x0903, 0x0901}:
                mod += chars[i]; i += 1

            unit += dep + mod
            guru = (dep in GURU_DEP) or bool(
                mod and (ANUSVARA in mod or VISARGA in mod))
            syllables.append({'text': unit, 'matras': 2 if guru else 1})

        else:
            i += 1   # punctuation, spaces, separators — skip

    # ── Fix C: second pass — laghu before conjunct becomes guru ──────────
    for k in range(len(syllables) - 1):
        if syllables[k]['matras'] != 1:
            continue
        nxt = syllables[k + 1]['text']
        has_internal_halant = HALANT in nxt[:-1]   # halant not at final pos
        next_is_zero        = (syllables[k + 1]['matras'] == 0)
        if has_internal_halant or next_is_zero:
            syllables[k] = dict(syllables[k], matras=2)

    return syllables


def total_matras(text: str) -> int:
    return sum(s['matras'] for s in syllabify(text))


# ════════════════════════════════════════════════════════════════════════════
# 2.  SPAN-LEVEL MATRA UTILITIES
# ════════════════════════════════════════════════════════════════════════════

def _is_continuation(token_id: int) -> bool:
    return CONT_LOW <= token_id <= CONT_HIGH


def _span_to_char(token_ids: list) -> str | None:
    try:
        raw = bytes(tid - BYT5_BYTE_OFFSET for tid in token_ids)
        return raw.decode('utf-8')
    except (UnicodeDecodeError, ValueError):
        return None


def _extract_char_spans(token_ids: list) -> list:
    """
    Return (start, end) pairs (exclusive end) for each character span.

    A new character starts at every non-continuation, non-special token.
    Continuation tokens (131–194) are absorbed into the preceding span.

    Examples:
      'मा' → [227,167,177, 227,167,193] → [(0,3), (3,6)]
      'सत्य' → [...18 tokens...] → [(0,3),(3,6),(6,9),(9,12)]
               decoded chars: 'स','त','्','य'
    """
    spans = []
    i, n = 0, len(token_ids)
    while i < n:
        tid = token_ids[i]
        if tid in SPECIAL or _is_continuation(tid):
            i += 1
            continue
        j = i + 1
        while j < n and _is_continuation(token_ids[j]):
            j += 1
        spans.append((i, j))
        i = j
    return spans


def _build_char_matra_table(token_ids: list,
                             charan_text: str | None = None) -> dict:
    """
    Build {token_position → matra_value} for every character-start position.

    Fix D — correct syllable-to-span alignment:

    The core challenge: syllabify() groups codepoints into syllable units
    (e.g. 'सत्य' → syllables ['सत्', 'य']), but _extract_char_spans() sees
    each Unicode codepoint as a separate token-span ('स','त','्','य').

    Strategy:
      1. Syllabify the full charan_text (context-aware, Fix C applies).
      2. For each syllable, the matra weight is assigned to the FIRST
         codepoint of the syllable. All other codepoints in the syllable
         (dependent vowels, halant, anusvara, conjunct members) get 0.0.
         Rationale: the first codepoint is always the consonant or vowel
         that heads the syllable; the model's choice of what to emit at
         that position is where the matra gradient should land.
      3. Walk the token spans in order, matching each span's decoded
         character to the next codepoint in the syllable sequence, and
         assign the pre-computed weight.

    Fix for "last non-halant" bug in prior version:
        The old code assigned matra to the last non-halant codepoint of
        the syllable (e.g. 'ā' in 'mā'), which has no independent token
        in many cases and misdirects the gradient. First-codepoint is
        always the syllable head and always has a token span.
    """
    # Reconstruct string from tokens when not provided
    if charan_text is None:
        spans = _extract_char_spans(token_ids)
        charan_text = ''.join(
            _span_to_char(token_ids[s:e]) or '' for s, e in spans)

    # Syllabify the full string — conjunct / anusvara rules apply (Fix C)
    syllables = syllabify(charan_text)

    # Build a flat list: one weight per Unicode codepoint of the full text.
    # Weight = syllable.matras for the FIRST codepoint of each syllable,
    #          0.0 for every subsequent codepoint within that syllable.
    char_weights: list[float] = []
    for syl in syllables:
        nfc = unicodedata.normalize('NFC', syl['text'])
        for ci, _ in enumerate(nfc):
            char_weights.append(float(syl['matras']) if ci == 0 else 0.0)

    # Walk token spans, match each decoded character to char_weights in order
    spans  = _extract_char_spans(token_ids)
    table  = {}
    weight_idx = 0
    for start, end in spans:
        ch = _span_to_char(token_ids[start:end])
        if ch is None:
            table[start] = 0.0
            continue
        # Each span decodes to exactly one Unicode codepoint for valid UTF-8
        if weight_idx < len(char_weights):
            table[start] = char_weights[weight_idx]
            weight_idx  += 1
        else:
            table[start] = 0.0

    return table


# ════════════════════════════════════════════════════════════════════════════
# 3.  DIFFERENTIABLE EXPECTED MATRA
# ════════════════════════════════════════════════════════════════════════════

def _build_non_cont_mask(vocab_size: int,
                          dtype: torch.dtype,
                          device: torch.device) -> torch.Tensor:
    """
    (V,) mask: 1.0 for tokens that can START a new character (Fix E: clamped).
    """
    mask = torch.ones(vocab_size, dtype=dtype, device=device)
    for tid in SPECIAL:
        if tid < vocab_size:
            mask[tid] = 0.0
    lo = min(CONT_LOW,  vocab_size)
    hi = min(CONT_HIGH, vocab_size - 1)
    if lo <= hi:
        mask[lo:hi + 1] = 0.0
    return mask


def _build_cont_mask(vocab_size: int,
                     dtype: torch.dtype,
                     device: torch.device) -> torch.Tensor:
    """(V,) mask: 1.0 for valid UTF-8 continuation byte tokens (Fix E: clamped)."""
    mask = torch.zeros(vocab_size, dtype=dtype, device=device)
    lo = min(CONT_LOW,  vocab_size)
    hi = min(CONT_HIGH, vocab_size - 1)
    if lo <= hi:
        mask[lo:hi + 1] = 1.0
    return mask


def compute_expected_matra_span(
    logits: torch.Tensor,         # (T, V) float32, grad-attached
    matra_table: dict,            # {position: matra_value}
    seg_start: int,
    seg_end: int,
    non_cont_mask: torch.Tensor,  # (V,) — tokens that can start a char
    cont_mask: torch.Tensor,      # (V,) — valid continuation byte tokens
) -> torch.Tensor:
    """
    Differentiable expected matra for one charan segment.

    Fix A — continuation-byte gradient:
        Primary gradient at leading byte s:
            matra_val × P(valid_start_byte | pos=s)
        Small gradient at each continuation pos s+k:
            γ × P(valid_continuation | pos=s+k),  γ = CONT_GRAD_WEIGHT = 0.1

        This penalises garbage continuation bytes (which would make
        _span_to_char return None and silently drop the matra contribution).
    """
    T, V = logits.shape
    CONT_GRAD_WEIGHT = 0.1

    total_expected = logits.new_zeros(1).squeeze()
    sorted_starts  = sorted(p for p in matra_table if seg_start <= p < seg_end)

    for idx, pos in enumerate(sorted_starts):
        matra_val = matra_table[pos]
        if pos >= T:
            continue

        # Leading byte — primary matra gradient
        probs   = torch.softmax(logits[pos], dim=-1)
        p_start = (probs * non_cont_mask).sum()
        if matra_val != 0.0:
            total_expected = total_expected + matra_val * p_start

        # Continuation bytes — UTF-8 validity regularisation (Fix A)
        next_start = (sorted_starts[idx + 1]
                      if idx + 1 < len(sorted_starts) else seg_end)
        next_start = min(next_start, T)
        for cont_pos in range(pos + 1, next_start):
            if cont_pos >= T:
                break
            p_cont = (torch.softmax(logits[cont_pos], dim=-1) * cont_mask).sum()
            total_expected = total_expected + CONT_GRAD_WEIGHT * p_cont

    return total_expected


# ════════════════════════════════════════════════════════════════════════════
# 4.  DATASET
# ════════════════════════════════════════════════════════════════════════════

DOHA_TARGETS      = [13.0, 11.0, 13.0, 11.0]
CHARAN_SEPARATORS = [' | ', ' || ', ' | ', '']


def _charan_byte_bounds(charans: list, tokenizer, max_tgt: int) -> list:
    """
    Compute token-position boundaries [0, e1, e2, e3, e4] for 4 charans.
    Fix B: called once at dataset load time — not inside collate_fn.
    """
    bounds = [0]
    pos    = 0
    for k, charan in enumerate(charans):
        pos += len(tokenizer.encode(charan, add_special_tokens=False))
        sep  = CHARAN_SEPARATORS[k]
        if sep:
            pos += len(tokenizer.encode(sep, add_special_tokens=False))
        bounds.append(min(pos, max_tgt))
    return bounds


class DohaDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer,
                 max_src: int = 256, max_tgt: int = 200):
        self.records = []
        for _, row in df.iterrows():
            try:
                charans = (ast.literal_eval(row['charans'])
                           if isinstance(row['charans'], str)
                           else row['charans'])
                matras  = (ast.literal_eval(row['matras'])
                           if isinstance(row['matras'], str)
                           else row['matras'])
            except Exception:
                charans, matras = [], []

            valid = bool(row.get('valid', False)) and len(charans) == 4

            # Fix B: pre-compute seg_bounds at load time
            seg_bounds = (
                _charan_byte_bounds(charans, tokenizer, max_tgt)
                if valid else None
            )

            self.records.append({
                'input':      f"विषय: {row['theme']} | संदर्भ: {row['context']}",
                'target':     str(row['formatted_doha']),
                'charans':    charans if valid else [],
                'matras':     matras,
                'valid':      valid,
                'seg_bounds': seg_bounds,
            })

    def __len__(self):          return len(self.records)
    def __getitem__(self, idx): return self.records[idx]


def collate_fn(batch, tokenizer, max_src: int = 256, max_tgt: int = 200):
    enc = tokenizer(
        [b['input']  for b in batch],
        padding=True, truncation=True,
        max_length=max_src, return_tensors='pt',
    )
    dec = tokenizer(
        text_target=[b['target'] for b in batch],
        padding=True, truncation=True,
        max_length=max_tgt, return_tensors='pt',
    )
    labels = dec.input_ids.clone()
    if dec.attention_mask is not None:
        labels[dec.attention_mask == 0] = -100

    return {
        'input_ids':      enc.input_ids,
        'attention_mask': enc.attention_mask,
        'labels':         labels,
        'seg_bounds':     [b['seg_bounds'] for b in batch],
        'charans':        [b['charans']    for b in batch],
        'gold_matras':    [b['matras']     for b in batch],
        'valid':          [b['valid']      for b in batch],
    }


# ════════════════════════════════════════════════════════════════════════════
# 5.  DIAGNOSTICS
# ════════════════════════════════════════════════════════════════════════════

def diagnose_all(df, train_loader, tokenizer, model):
    print("=" * 60)
    print("CSV DIAGNOSTICS")
    print("=" * 60)
    print(f"Shape  : {df.shape}")
    print(f"Nulls  :\n{df[['theme','context','formatted_doha','charans','matras','valid']].isnull().sum()}")
    print(f"Valid  : {df['valid'].value_counts().to_dict()}")

    print("\n" + "=" * 60)
    print("MODEL / TOKENIZER DIAGNOSTICS  (Fix E)")
    print("=" * 60)
    emb_shape = model.model.shared.weight.shape
    actual_V  = emb_shape[0]
    print(f"Embedding shape  : {emb_shape}")
    print(f"tokenizer vocab  : {tokenizer.vocab_size}")
    print(f"VOCAB_SIZE const : {VOCAB_SIZE}  model.vocab_size: {model.vocab_size}")
    print(f"CONT range       : [{CONT_LOW}, {min(CONT_HIGH, actual_V - 1)}]")
    if actual_V < VOCAB_SIZE:
        print(f"  ⚠ Embedding rows ({actual_V}) < VOCAB_SIZE — clamped to {actual_V}")
    else:
        print(f"  ✓ Embedding rows ≥ VOCAB_SIZE constant")

    sample   = 'म'
    enc_ids  = tokenizer.encode(sample, add_special_tokens=False)
    expected = [b + BYT5_BYTE_OFFSET for b in sample.encode('utf-8')]
    print(f"Offset check 'म': encoded={enc_ids}  expected={expected}  "
          f"✓={enc_ids == expected}")

    print("\n" + "=" * 60)
    print("BATCH DIAGNOSTICS")
    print("=" * 60)
    batch      = next(iter(train_loader))
    labels     = batch['labels']
    non_masked = labels != -100
    visible    = labels[non_masked]
    print(f"input_ids shape : {batch['input_ids'].shape}")
    print(f"labels shape    : {labels.shape}")
    print(f"Non-masked      : {non_masked.sum().item()} / {labels.numel()} "
          f"({100*non_masked.float().mean():.1f}%)")
    print(f"Label id range  : {visible.min().item()} – {visible.max().item()}")
    print(f"Embedding rows  : {actual_V}")
    if visible.max().item() >= actual_V:
        print("  ✗ LABEL ID EXCEEDS EMBEDDING TABLE — will cause nan!")
    else:
        print("  ✓ All label ids within embedding range")
    print(f"\nDecoded sample  : {tokenizer.decode(visible[:60])}")

    model.eval()
    with torch.no_grad():
        raw_out = model.model(
            input_ids      = batch['input_ids'].to(device),
            attention_mask = batch['attention_mask'].to(device),
            labels         = batch['labels'].to(device),
        )
    loss_val = raw_out.loss.item()
    print(f"\nFloat32 CE (raw model, no AMP) : {loss_val:.6f}")
    print(f"Is finite                      : {math.isfinite(loss_val)}")
    if not math.isfinite(loss_val):
        print("  ✗ Still nan without AMP → deeper issue")
    else:
        print("  ✓ Finite without AMP → AMP overflow was the only problem")

    # ── Fix C: syllabify spot-checks ─────────────────────────────────────
    # Expected values reflect actual Hindi prosody:
    #   'राम'   → 'रा'(guru=2) + 'म'(laghu=1) = [2, 1]
    #             'ा' is in GURU_DEP, so 'रा' is guru. [1,1,1] was wrong.
    #   'मात्र' → 'मा'(guru=2) + 'त्र'(conjunct=0, absorbed into one unit) ...
    #             syllabify gives 'मा'(2) + 'त्र'(0) = [2, 0].
    #             'त्र' is a conjunct so matra=0 for that unit. [1,2,1] was wrong.
    #   'सत्'   → 'स'(laghu before halant→guru=2) + 'त्'(halant unit=0) = [2, 0].
    #             The zero-matra halant unit IS emitted by syllabify. [2] was wrong.
    print("\n" + "=" * 60)
    print("SYLLABIFY / HALANT DIAGNOSTICS  (Fix C)")
    print("=" * 60)
    halant_cases = [
        ('सत्य',  [2, 1],    "स(laghu→guru before त्य conjunct), य(laghu)"),
        ('धर्म',  [2, 1],    "ध(laghu→guru before र्म conjunct), म(laghu)"),
        ('कर्म',  [2, 1],    "क(laghu→guru before र्म), म(laghu)"),
        ('राम',   [2, 1],    "रा(guru — ā ∈ GURU_DEP), म(laghu)"),
        ('मात्र', [2, 0],    "मा(guru), त्र(conjunct → zero matra)"),
        ('सत्',   [2, 0],    "स(laghu→guru before त् halant unit), त्(zero)"),
    ]
    all_ok = True
    for text, expected_matras, note in halant_cases:
        syls   = syllabify(text)
        got    = [s['matras'] for s in syls]
        ok     = (got == expected_matras)
        status = '✓' if ok else '✗'
        if not ok:
            all_ok = False
        print(f"  {status} '{text}': got {got}  expected {expected_matras}  ({note})")
    if not all_ok:
        print("  ⚠ Some cases differ — review syllabify() conjunct rules")
    else:
        print("  ✓ All syllabify cases correct")

    # ── Fix D: span matra table spot-checks ──────────────────────────────
    # Expected: matra weight on the FIRST (head) codepoint of each syllable.
    #   'मा'   → 'म'(2.0), 'ā'(0.0)     — 'म' heads the syllable
    #   'सत्य' → 'स'(2.0), 'त'(0.0), '्'(0.0), 'य'(1.0)
    #   'राम'  → 'र'(2.0), 'ā'(0.0), 'म'(1.0)
    #   'कर्म' → 'क'(2.0), 'र'(0.0), '्'(0.0), 'म'(1.0)
    print("\n" + "=" * 60)
    print("SPAN MATRA TABLE DIAGNOSTICS  (Fix D)")
    print("=" * 60)
    span_expected = {
        'मा':   {'म': 2.0, 'ा': 0.0},
        'सत्य': {'स': 2.0, 'त': 0.0, '्': 0.0, 'य': 1.0},
        'राम':  {'र': 2.0, 'ा': 0.0, 'म': 1.0},
        'कर्म': {'क': 2.0, 'र': 0.0, '्': 0.0, 'म': 1.0},
    }
    all_ok = True
    for text, exp_map in span_expected.items():
        tids  = tokenizer.encode(text, add_special_tokens=False)
        table = _build_char_matra_table(tids, charan_text=text)
        spans = _extract_char_spans(tids)
        print(f"  '{text}' → token_ids={tids}")
        for s, e in spans:
            ch     = _span_to_char(tids[s:e]) or '?'
            got    = table.get(s, '?')
            exp    = exp_map.get(ch, '?')
            ok     = (got == exp) if exp != '?' else True
            status = '✓' if ok else '✗'
            if not ok:
                all_ok = False
            print(f"    {status} span [{s}:{e}] = '{ch}'  "
                  f"matra={got}  expected={exp}")
    if not all_ok:
        print("  ⚠ Some matra table values differ — review _build_char_matra_table()")
    else:
        print("  ✓ All span matra table values correct")

    print("=" * 60 + "\n")
    model.train()


# ════════════════════════════════════════════════════════════════════════════
# 6.  MODEL
# ════════════════════════════════════════════════════════════════════════════

class DohaByT5(nn.Module):
    def __init__(self, model_name: str, lambda_matra: float = 0.0,
                 vocab_size: int = VOCAB_SIZE):
        super().__init__()
        self.model = T5ForConditionalGeneration.from_pretrained(model_name)

        # Fix E: clamp to actual embedding rows
        actual_V = self.model.shared.weight.shape[0]
        if vocab_size > actual_V:
            print(f"  ⚠ vocab_size={vocab_size} > actual embedding rows={actual_V}."
                  f" Using {actual_V}.")
        self.vocab_size   = min(vocab_size, actual_V)
        self.lambda_matra = lambda_matra

        self.register_buffer(
            'targets',
            torch.tensor(DOHA_TARGETS, dtype=torch.float32)
        )

        # Pre-build masks on CPU; transferred to device/dtype on demand
        self._non_cont_mask_cpu = _build_non_cont_mask(
            self.vocab_size, torch.float32, torch.device('cpu'))
        self._cont_mask_cpu     = _build_cont_mask(
            self.vocab_size, torch.float32, torch.device('cpu'))

    def _masks(self, dtype, dev):
        return (self._non_cont_mask_cpu.to(dtype=dtype, device=dev),
                self._cont_mask_cpu.to(dtype=dtype, device=dev))

    def compute_matra_loss(self,
                           logits: torch.Tensor,
                           labels: torch.Tensor,
                           seg_bounds: list,
                           charans_batch: list) -> torch.Tensor:
        B, T, V        = logits.shape
        tgts           = self.targets.to(dtype=logits.dtype, device=logits.device)
        non_cont, cont = self._masks(logits.dtype, logits.device)

        seg_loss_list = []

        for b in range(B):
            bounds  = seg_bounds[b]
            charans = charans_batch[b]
            if bounds is None or len(bounds) < 5:
                continue

            label_ids = labels[b].tolist()
            clean_ids = [tid if tid != -100 else 0 for tid in label_ids]

            sample_losses = []
            for k in range(4):
                s, e = bounds[k], bounds[k + 1]
                if s >= e or e > T:
                    continue

                charan_text = charans[k] if k < len(charans) else None

                # Build table on segment slice; remap keys to full positions
                local_table = _build_char_matra_table(
                    clean_ids[s:e], charan_text=charan_text)
                matra_table = {s + lp: mv for lp, mv in local_table.items()}

                M_hat_k = compute_expected_matra_span(
                    logits[b], matra_table,
                    seg_start=s, seg_end=e,
                    non_cont_mask=non_cont, cont_mask=cont,
                )

                sample_losses.append(
                    F.smooth_l1_loss(M_hat_k, tgts[k], reduction='sum'))

            if sample_losses:
                seg_loss_list.append(torch.stack(sample_losses).mean())

        if seg_loss_list:
            return torch.stack(seg_loss_list).mean()
        return (logits * 0.0).sum()

    def forward(self, input_ids, attention_mask, labels,
                seg_bounds, charans, gold_matras, **kwargs) -> dict:
        with torch.amp.autocast('cuda', enabled=False):
            outputs = self.model(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        logits = outputs.logits.float()

        active = labels != -100
        if active.any():
            L_ce = F.cross_entropy(
                logits[active], labels[active],
                label_smoothing=0.1, reduction='mean')
        else:
            L_ce = logits.new_zeros(1).squeeze()

        if not torch.isfinite(L_ce):
            print(f"  ⚠ CE non-finite ({L_ce.item()}), zeroing.")
            L_ce = logits.new_zeros(1).squeeze()

        if self.lambda_matra > 0.0:
            L_matra = self.compute_matra_loss(logits, labels, seg_bounds, charans)
            L_total = L_ce + self.lambda_matra * L_matra
        else:
            L_matra = logits.new_zeros(1).squeeze()
            L_total = L_ce

        return {
            'loss':       L_total,
            'ce_loss':    L_ce.detach(),
            'matra_loss': L_matra.detach(),
        }


# ════════════════════════════════════════════════════════════════════════════
# 7.  TRAINING
# ════════════════════════════════════════════════════════════════════════════

def train():
    os.makedirs(OUTPUT_DIR,      exist_ok=True)
    os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

    print(f"Loading tokenizer and model from '{STAGE1_MODEL_ID}'...")
    tokenizer = AutoTokenizer.from_pretrained(STAGE1_MODEL_ID)
    model     = DohaByT5(STAGE1_MODEL_ID, lambda_matra=0.0,
                         vocab_size=VOCAB_SIZE).to(device)
    print(f"✓ Model loaded — embedding: {model.model.shared.weight.shape} "
          f"| effective vocab_size: {model.vocab_size}")

    df      = pd.read_csv(CSV_PATH)
    full_ds = DohaDataset(df, tokenizer, MAX_SRC, MAX_TGT)
    val_n   = int(len(full_ds) * VAL_SPLIT)
    train_ds, val_ds = random_split(
        full_ds, [len(full_ds) - val_n, val_n],
        generator=torch.Generator().manual_seed(SEED),
    )
    print(f"Train: {len(train_ds)} | Val: {val_n}")

    _col         = lambda b: collate_fn(b, tokenizer, MAX_SRC, MAX_TGT)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=_col, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              collate_fn=_col, num_workers=2, pin_memory=True)

    diagnose_all(df, train_loader, tokenizer, model)

    optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps  = (len(train_loader) // GRAD_ACCUM_STEPS) * NUM_EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, warmup_steps, total_steps)

    scaler = torch.amp.GradScaler('cuda')

    best_val_loss = float('inf')
    history       = []

    for epoch in range(1, NUM_EPOCHS + 1):

        if epoch == 1:
            model.lambda_matra = 0.0
        elif epoch <= LAMBDA_WARMUP_EPOCHS:
            model.lambda_matra = LAMBDA_MAX * (epoch - 1) / (LAMBDA_WARMUP_EPOCHS - 1)
        else:
            model.lambda_matra = LAMBDA_MAX

        # ── Train ────────────────────────────────────────────────────────
        model.train()
        tr_ce = tr_matra = tr_total = 0.0
        nan_count = 0
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader):
            out = model(
                input_ids      = batch['input_ids'].to(device),
                attention_mask = batch['attention_mask'].to(device),
                labels         = batch['labels'].to(device),
                seg_bounds     = batch['seg_bounds'],
                charans        = batch['charans'],
                gold_matras    = batch['gold_matras'],
            )
            loss = out['loss'] / GRAD_ACCUM_STEPS

            if not torch.isfinite(out['loss']):
                nan_count += 1
                optimizer.zero_grad()
                continue

            scaler.scale(loss).backward()

            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                if not math.isfinite(grad_norm):
                    optimizer.zero_grad()
                    scaler.update()
                    continue
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

            tr_ce    += out['ce_loss'].item()
            tr_matra += out['matra_loss'].item()
            tr_total += out['loss'].item()

            if (step + 1) % 50 == 0:
                print(f"  Ep {epoch:02d} | Step {step+1:04d}/{len(train_loader)} "
                      f"| CE {tr_ce/(step+1):.4f} "
                      f"| Matra {tr_matra/(step+1):.4f} "
                      f"| λ={model.lambda_matra:.3f}")

        n      = len(train_loader)
        avg_tr = {'ce': tr_ce/n, 'matra': tr_matra/n, 'total': tr_total/n}
        if nan_count:
            print(f"  ⚠ {nan_count} nan steps skipped this epoch")

        # ── Validate ─────────────────────────────────────────────────────
        model.eval()
        va_ce = va_matra = va_total = 0.0
        with torch.no_grad():
            for batch in val_loader:
                out = model(
                    input_ids      = batch['input_ids'].to(device),
                    attention_mask = batch['attention_mask'].to(device),
                    labels         = batch['labels'].to(device),
                    seg_bounds     = batch['seg_bounds'],
                    charans        = batch['charans'],
                    gold_matras    = batch['gold_matras'],
                )
                va_ce    += out['ce_loss'].item()
                va_matra += out['matra_loss'].item()
                va_total += out['loss'].item()

        m      = len(val_loader)
        avg_va = {'ce': va_ce/m, 'matra': va_matra/m, 'total': va_total/m}

        print(f"\nEpoch {epoch:02d} | λ={model.lambda_matra:.3f}")
        print(f"  Train → CE:{avg_tr['ce']:.4f} Matra:{avg_tr['matra']:.4f} "
              f"Total:{avg_tr['total']:.4f}")
        print(f"  Val   → CE:{avg_va['ce']:.4f} Matra:{avg_va['matra']:.4f} "
              f"Total:{avg_va['total']:.4f}\n")

        history.append({'epoch': epoch, 'train': avg_tr,
                        'val': avg_va, 'lambda': model.lambda_matra})

        if avg_va['total'] < best_val_loss:
            best_val_loss = avg_va['total']
            ckpt          = os.path.join(OUTPUT_DIR, 'best_model')
            model.model.save_pretrained(ckpt)
            tokenizer.save_pretrained(ckpt)
            torch.save({'lambda_matra': model.lambda_matra, 'epoch': epoch},
                       os.path.join(ckpt, 'meta.pt'))
            print(f"  ✓ Saved best (val_loss={best_val_loss:.4f})")

        with open(os.path.join(OUTPUT_DIR, 'history.json'), 'w',
                  encoding='utf-8') as f:
            json.dump(history, f, indent=2, ensure_ascii=False)

    # ── Final save & push ─────────────────────────────────────────────────
    model.model.save_pretrained(FINAL_MODEL_DIR)
    tokenizer.save_pretrained(FINAL_MODEL_DIR)
    torch.save({'lambda_matra': model.lambda_matra, 'epoch': NUM_EPOCHS},
               os.path.join(FINAL_MODEL_DIR, 'meta.pt'))

    if HF_TOKEN:
        api = HfApi()
        api.create_repo(repo_id=STAGE2_HUB_ID, exist_ok=True, token=HF_TOKEN)
        api.upload_folder(
            folder_path    = os.path.join(OUTPUT_DIR, 'best_model'),
            repo_id        = STAGE2_HUB_ID,
            commit_message = f"Stage 2 — val_loss={best_val_loss:.4f}, span matra loss",
            token          = HF_TOKEN,
        )
        print(f"✓ Pushed → https://huggingface.co/{STAGE2_HUB_ID}")

    return model, tokenizer


# ════════════════════════════════════════════════════════════════════════════
# 8.  INFERENCE
# ════════════════════════════════════════════════════════════════════════════

def _parse_formatted(text: str) -> list:
    text  = text.replace('||', '|')
    parts = [p.strip() for p in text.split('|')]
    return [p for p in parts if p]


def generate_doha(model, tokenizer, theme: str, context: str = '',
                  num_beams: int = 10, max_new_tokens: int = 150,
                  tolerance: int = 1) -> dict:
    """Fix B: repetition_penalty=1.3 prevents '||' separator loops."""
    TARGETS = [13, 11, 13, 11]
    inputs  = tokenizer(
        f"विषय: {theme} | संदर्भ: {context}",
        return_tensors='pt').to(device)

    model.eval()
    with torch.no_grad():
        out = model.model.generate(
            **inputs,
            num_beams            = num_beams,
            num_return_sequences = num_beams,
            max_new_tokens       = max_new_tokens,
            early_stopping       = True,
            repetition_penalty   = 1.3,
        )

    candidates  = tokenizer.batch_decode(out, skip_special_tokens=True)
    best, best_score, best_matras = None, float('inf'), []

    for cand in candidates:
        charans = _parse_formatted(cand)
        if len(charans) != 4:
            continue
        m     = [total_matras(c) for c in charans]
        score = sum(abs(m[k] - TARGETS[k]) for k in range(4))
        if score < best_score:
            best_score, best, best_matras = score, cand, m

    return {
        'doha':           best or candidates[0],
        'matras':         best_matras,
        'matra_error':    best_score,
        'valid':          best_score <= tolerance * 4,
        'all_candidates': candidates,
    }


# ════════════════════════════════════════════════════════════════════════════
# 9.  ENTRY POINT
# ════════════════════════════════════════════════════════════════════════════

if __name__ == '__main__':
    model, tokenizer = train()

    result = generate_doha(
        model, tokenizer,
        theme   = 'शृंगार',
        context = 'नायिका की सुंदरता का वर्णन',
    )
    print("\nGenerated Doha:")
    print(result['doha'])
    print(f"Matras      : {result['matras']}")
    print(f"Matra error : {result['matra_error']}")
    print(f"Valid       : {result['valid']}")

model_type         : t5
config.vocab_size  : 384
tokenizer type     : ByT5Tokenizer
tokenizer.vocab_size: 256

First 20 tokens:
  [  0] '<pad>'
  [  1] '</s>'
  [  2] '<unk>'
  [  3] '\x00'
  [  4] '\x01'
  [  5] '\x02'
  [  6] '\x03'
  [  7] '\x04'
  [  8] '\x05'
  [  9] '\x06'
  [ 10] '\x07'
  [ 11] '\x08'
  [ 12] '\t'
  [ 13] '\n'
  [ 14] '\x0b'
  [ 15] '\x0c'
  [ 16] '\r'
  [ 17] '\x0e'
  [ 18] '\x0f'
  [ 19] '\x10'

Last 20 tokens:
  [364] '<extra_id_105>'
  [365] '<extra_id_106>'
  [366] '<extra_id_107>'
  [367] '<extra_id_108>'
  [368] '<extra_id_109>'
  [369] '<extra_id_110>'
  [370] '<extra_id_111>'
  [371] '<extra_id_112>'
  [372] '<extra_id_113>'
  [373] '<extra_id_114>'
  [374] '<extra_id_115>'
  [375] '<extra_id_116>'
  [376] '<extra_id_117>'
  [377] '<extra_id_118>'
  [378] '<extra_id_119>'
  [379] '<extra_id_120>'
  [380] '<extra_id_121>'
  [381] '<extra_id_122>'
  [382] '<extra_id_123>'
  [383] '<extra_id_124>'

Sample: मोर पच्छ जो सिर चढ़ै | बारन तें अधिकाय
Token ids :

In [1]:
"""
Standalone Inference Script for Theme-conditioned Doha Generation.
Loads the Stage 2 model from Hugging Face and evaluates matra constraints.
"""

import unicodedata
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration

# ════════════════════════════════════════════════════════════════════════════
# 0. SETUP & CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════

MODEL_ID = "KGan31/Doha-Gen-Stage2_Matra_loss_integrated"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Loading tokenizer and model from: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = T5ForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
print(f"✓ Model loaded to {device}\n")

# ════════════════════════════════════════════════════════════════════════════
# 1. MATRA UTILITIES (Extracted from training script)
# ════════════════════════════════════════════════════════════════════════════

HALANT, ANUSVARA, VISARGA, NUKTA = '\u094D', '\u0902', '\u0903', '\u093C'
CONSONANTS = set(range(0x0915, 0x093A)) | set(range(0x0958, 0x0960))
IND_VOWELS = set(range(0x0905, 0x0915))
DEP_VOWELS = set(range(0x093E, 0x094D)) | {0x094E, 0x094F}
GURU_DEP   = {'\u093E', '\u0940', '\u0942', '\u0947', '\u0948', '\u094B', '\u094C', '\u094F'}
GURU_IND   = {'\u0906', '\u0908', '\u090A', '\u090F', '\u0910', '\u0913', '\u0914', '\u0960'}
DEVANAGARI_DIGIT_MATRAS = {chr(0x0966 + i): v for i, v in enumerate([2, 1, 2, 1, 2, 2, 2, 2, 2, 2])}
ARABIC_DIGIT_MATRAS = {str(d): 2 for d in range(10)}

def syllabify(text: str) -> list:
    text = unicodedata.normalize('NFC', text)
    chars, n, syllables, i = list(text), len(text), [], 0

    while i < n:
        ch, cp = chars[i], ord(chars[i])

        if cp in range(0x0966, 0x0970):
            syllables.append({'text': ch, 'matras': DEVANAGARI_DIGIT_MATRAS.get(ch, 2)})
            i += 1
        elif ch.isdigit():
            syllables.append({'text': ch, 'matras': ARABIC_DIGIT_MATRAS.get(ch, 2)})
            i += 1
        elif cp in IND_VOWELS:
            unit = ch; i += 1
            while i < n and ord(chars[i]) in {0x0902, 0x0903, 0x0901}:
                unit += chars[i]; i += 1
            guru = ch in GURU_IND or ANUSVARA in unit or VISARGA in unit
            syllables.append({'text': unit, 'matras': 2 if guru else 1})
        elif cp in CONSONANTS:
            unit = ch; i += 1
            if i < n and chars[i] == NUKTA:
                unit += chars[i]; i += 1
            while (i + 1 < n and chars[i] == HALANT and ord(chars[i + 1]) in CONSONANTS):
                unit += chars[i] + chars[i + 1]; i += 2
                if i < n and chars[i] == NUKTA:
                    unit += chars[i]; i += 1
            if i < n and chars[i] == HALANT:
                unit += chars[i]; i += 1
                syllables.append({'text': unit, 'matras': 0})
                continue
            dep = ''
            if i < n and ord(chars[i]) in DEP_VOWELS:
                dep = chars[i]; i += 1
            mod = ''
            while i < n and ord(chars[i]) in {0x0902, 0x0903, 0x0901}:
                mod += chars[i]; i += 1
            unit += dep + mod
            guru = (dep in GURU_DEP) or bool(mod and (ANUSVARA in mod or VISARGA in mod))
            syllables.append({'text': unit, 'matras': 2 if guru else 1})
        else:
            i += 1  

    # FIX C: second pass — laghu before conjunct cluster becomes guru
    for k in range(len(syllables) - 1):
        if syllables[k]['matras'] != 1:
            continue
        nxt = syllables[k + 1]['text']
        has_internal_halant = HALANT in nxt[:-1]
        next_is_zero        = (syllables[k + 1]['matras'] == 0)
        if has_internal_halant or next_is_zero:
            syllables[k] = dict(syllables[k], matras=2)

    return syllables

def total_matras(text: str) -> int:
    return sum(s['matras'] for s in syllabify(text))

# ════════════════════════════════════════════════════════════════════════════
# 2. GENERATION LOGIC
# ════════════════════════════════════════════════════════════════════════════

def _parse_formatted(text: str) -> list:
    text  = text.replace('||', '|')
    parts = [p.strip() for p in text.split('|')]
    return [p for p in parts if p]

def generate_doha(theme: str, context: str = '', num_beams: int = 10, max_new_tokens: int = 400, tolerance: int = 1) -> dict:
    TARGETS = [13, 11, 13, 11]
    inputs = tokenizer(f"विषय: {theme} | संदर्भ: {context}", return_tensors='pt').to(device)

    model.eval()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            do_sample=True,
            top_p=0.85,           # Tighter nucleus: only consider top 85% of byte probabilities
            temperature=0.3,      # Drastically lowered from 0.8. Keeps spelling and grammar strict.
            num_return_sequences=num_beams,
            max_new_tokens=max_new_tokens,
            early_stopping=True,
        )

    candidates = tokenizer.batch_decode(out, skip_special_tokens=True)
    best, best_score, best_matras = None, float('inf'), []

    for cand in candidates:
        charans = _parse_formatted(cand)
        if len(charans) != 4:
            continue
        
        m = [total_matras(c) for c in charans]
        score = sum(abs(m[k] - TARGETS[k]) for k in range(4))
        
        if score < best_score:
            best_score, best, best_matras = score, cand, m

    # Fallback to the top candidate if none had exactly 4 charans
    if best is None:
        best = candidates[0]
        charans = _parse_formatted(best)
        best_matras = [total_matras(c) for c in charans]
        best_score = sum(abs(m - t) for m, t in zip(best_matras, TARGETS)) if len(best_matras) == 4 else 999

    return {
        'doha': best,
        'matras': best_matras,
        'matra_error': best_score,
        'valid': best_score <= tolerance * 4 and len(best_matras) == 4,
        'all_candidates': candidates,
    }

# ════════════════════════════════════════════════════════════════════════════
# 3. RUN TESTS
# ════════════════════════════════════════════════════════════════════════════

if __name__ == '__main__':
    test_cases = [
        {"theme": "शृंगार", "context": "नायिका की सुंदरता का वर्णन"},
        {"theme": "भक्ति", "context": "ईश्वर के प्रति समर्पण"},
        {"theme": "नीति", "context": "सज्जन पुरुषों का स्वभाव"},
        {"theme": "प्रकृति", "context": "वर्षा ऋतु का सुंदर दृश्य"}
    ]

    print("="*60)
    print(" DOHA GENERATION TESTS (Target: [13, 11, 13, 11])")
    print("="*60)

    for i, test in enumerate(test_cases, 1):
        print(f"\n--- Test {i} ---")
        print(f"Theme   : {test['theme']}")
        print(f"Context : {test['context']}")
        
        result = generate_doha(theme=test['theme'], context=test['context'])
        
        print(f"\nResult:")
        print(result['doha'])
        print(f"Matras      : {result['matras']}")
        print(f"Matra Error : {result['matra_error']}")
        
        status = "✅ PASS" if result['valid'] else "❌ FAIL"
        print(f"Validation  : {status}")

Loading tokenizer and model from: KGan31/Doha-Gen-Stage2_Matra_loss_integrated


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✓ Model loaded to cuda

 DOHA GENERATION TESTS (Target: [13, 11, 13, 11])

--- Test 1 ---
Theme   : शृंगार
Context : नायिका की सुंदरता का वर्णन

Result:
सब का करता की सुंदरता || साधन की काम का काम || सब कोई नायिका की सुंदरता का वर्णन
Matras      : [16, 14, 25]
Matra Error : 999
Validation  : ❌ FAIL

--- Test 2 ---
Theme   : भक्ति
Context : ईश्वर के प्रति समर्पण

Result:
मानव करते हैं पानी || क्या के प्रति समर्पण || क्या कोई प्रति समर्पण || क्या क्या क्या करते है
Matras      : [14, 11, 13, 12]
Matra Error : 2
Validation  : ✅ PASS

--- Test 3 ---
Theme   : नीति
Context : सज्जन पुरुषों का स्वभाव

Result:
पुरुष की पुरुष के साथ, | मन की साथ की सुख || पुरुष के साथ करता || समझन के साथ
Matras      : [13, 11, 12, 9]
Matra Error : 3
Validation  : ✅ PASS

--- Test 4 ---
Theme   : प्रकृति
Context : वर्षा ऋतु का सुंदर दृश्य

Result:
सब को साथ के साथ || साथ को सुंदर समय || साथ को समय की समय || करते निर्मल के साथ
Matras      : [12, 12, 13, 13]
Matra Error : 4
Validation  : ✅ PASS


In [5]:
"""
Standalone Inference Script for Theme-conditioned Doha Generation.
Loads the Stage 2 model from Hugging Face and evaluates matra constraints.
"""

import unicodedata
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration

# ════════════════════════════════════════════════════════════════════════════
# 0. SETUP & CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════

MODEL_ID = "KGan31/Doha-Gen-Stage2_Matra_loss_integrated"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Loading tokenizer and model from: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = T5ForConditionalGeneration.from_pretrained(MODEL_ID).to(device)
print(f"✓ Model loaded to {device}\n")

# ════════════════════════════════════════════════════════════════════════════
# 1. MATRA UTILITIES (Extracted from training script)
# ════════════════════════════════════════════════════════════════════════════

HALANT, ANUSVARA, VISARGA, NUKTA = '\u094D', '\u0902', '\u0903', '\u093C'
CONSONANTS = set(range(0x0915, 0x093A)) | set(range(0x0958, 0x0960))
IND_VOWELS = set(range(0x0905, 0x0915))
DEP_VOWELS = set(range(0x093E, 0x094D)) | {0x094E, 0x094F}
GURU_DEP   = {'\u093E', '\u0940', '\u0942', '\u0947', '\u0948', '\u094B', '\u094C', '\u094F'}
GURU_IND   = {'\u0906', '\u0908', '\u090A', '\u090F', '\u0910', '\u0913', '\u0914', '\u0960'}
DEVANAGARI_DIGIT_MATRAS = {chr(0x0966 + i): v for i, v in enumerate([2, 1, 2, 1, 2, 2, 2, 2, 2, 2])}
ARABIC_DIGIT_MATRAS = {str(d): 2 for d in range(10)}

def syllabify(text: str) -> list:
    text = unicodedata.normalize('NFC', text)
    chars, n, syllables, i = list(text), len(text), [], 0

    while i < n:
        ch, cp = chars[i], ord(chars[i])

        if cp in range(0x0966, 0x0970):
            syllables.append({'text': ch, 'matras': DEVANAGARI_DIGIT_MATRAS.get(ch, 2)})
            i += 1
        elif ch.isdigit():
            syllables.append({'text': ch, 'matras': ARABIC_DIGIT_MATRAS.get(ch, 2)})
            i += 1
        elif cp in IND_VOWELS:
            unit = ch; i += 1
            while i < n and ord(chars[i]) in {0x0902, 0x0903, 0x0901}:
                unit += chars[i]; i += 1
            guru = ch in GURU_IND or ANUSVARA in unit or VISARGA in unit
            syllables.append({'text': unit, 'matras': 2 if guru else 1})
        elif cp in CONSONANTS:
            unit = ch; i += 1
            if i < n and chars[i] == NUKTA:
                unit += chars[i]; i += 1
            while (i + 1 < n and chars[i] == HALANT and ord(chars[i + 1]) in CONSONANTS):
                unit += chars[i] + chars[i + 1]; i += 2
                if i < n and chars[i] == NUKTA:
                    unit += chars[i]; i += 1
            if i < n and chars[i] == HALANT:
                unit += chars[i]; i += 1
                syllables.append({'text': unit, 'matras': 0})
                continue
            dep = ''
            if i < n and ord(chars[i]) in DEP_VOWELS:
                dep = chars[i]; i += 1
            mod = ''
            while i < n and ord(chars[i]) in {0x0902, 0x0903, 0x0901}:
                mod += chars[i]; i += 1
            unit += dep + mod
            guru = (dep in GURU_DEP) or bool(mod and (ANUSVARA in mod or VISARGA in mod))
            syllables.append({'text': unit, 'matras': 2 if guru else 1})
        else:
            i += 1  

    # FIX C: second pass — laghu before conjunct cluster becomes guru
    for k in range(len(syllables) - 1):
        if syllables[k]['matras'] != 1:
            continue
        nxt = syllables[k + 1]['text']
        has_internal_halant = HALANT in nxt[:-1]
        next_is_zero        = (syllables[k + 1]['matras'] == 0)
        if has_internal_halant or next_is_zero:
            syllables[k] = dict(syllables[k], matras=2)

    return syllables

def total_matras(text: str) -> int:
    return sum(s['matras'] for s in syllabify(text))

# ════════════════════════════════════════════════════════════════════════════
# 2. GENERATION LOGIC
# ════════════════════════════════════════════════════════════════════════════

def _parse_formatted(text: str) -> list:
    text  = text.replace('||', '|')
    parts = [p.strip() for p in text.split('|')]
    return [p for p in parts if p]

def generate_doha(theme: str, context: str = '', num_beams: int = 10, max_new_tokens: int = 400, tolerance: int = 1) -> dict:
    TARGETS = [13, 11, 13, 11]
    inputs = tokenizer(f"विषय: {theme} | संदर्भ: {context}", return_tensors='pt').to(device)

    model.eval()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            do_sample=True,
            top_p=0.90,               # Slightly wider nucleus
            temperature=0.6,          # Enough heat to encourage richer vocabulary
            no_repeat_ngram_size=12,  # The Magic Fix: Prevents repeating ~3-4 character sequences (words/phrases)
            num_return_sequences=num_beams,
            max_new_tokens=max_new_tokens,
            early_stopping=True,
        )

    candidates = tokenizer.batch_decode(out, skip_special_tokens=True)
    best, best_score, best_matras = None, float('inf'), []

    for cand in candidates:
        charans = _parse_formatted(cand)
        if len(charans) != 4:
            continue
        
        m = [total_matras(c) for c in charans]
        score = sum(abs(m[k] - TARGETS[k]) for k in range(4))
        
        if score < best_score:
            best_score, best, best_matras = score, cand, m

    # Fallback to the top candidate if none had exactly 4 charans
    if best is None:
        best = candidates[0]
        charans = _parse_formatted(best)
        best_matras = [total_matras(c) for c in charans]
        best_score = sum(abs(m - t) for m, t in zip(best_matras, TARGETS)) if len(best_matras) == 4 else 999

    return {
        'doha': best,
        'matras': best_matras,
        'matra_error': best_score,
        'valid': best_score <= tolerance * 4 and len(best_matras) == 4,
        'all_candidates': candidates,
    }

# ════════════════════════════════════════════════════════════════════════════
# 3. RUN TESTS
# ════════════════════════════════════════════════════════════════════════════

if __name__ == '__main__':
    test_cases = [
        {"theme": "शृंगार", "context": "नायिका की सुंदरता का वर्णन"},
        {"theme": "भक्ति", "context": "ईश्वर के प्रति समर्पण"},
        {"theme": "नीति", "context": "सज्जन पुरुषों का स्वभाव"},
        {"theme": "प्रकृति", "context": "वर्षा ऋतु का सुंदर दृश्य"}
    ]

    print("="*60)
    print(" DOHA GENERATION TESTS (Target: [13, 11, 13, 11])")
    print("="*60)

    for i, test in enumerate(test_cases, 1):
        print(f"\n--- Test {i} ---")
        print(f"Theme   : {test['theme']}")
        print(f"Context : {test['context']}")
        
        result = generate_doha(theme=test['theme'], context=test['context'])
        
        print(f"\nResult:")
        print(result['doha'])
        print(f"Matras      : {result['matras']}")
        print(f"Matra Error : {result['matra_error']}")
        
        status = "✅ PASS" if result['valid'] else "❌ FAIL"
        print(f"Validation  : {status}")

Loading tokenizer and model from: KGan31/Doha-Gen-Stage2_Matra_loss_integrated


Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✓ Model loaded to cuda

 DOHA GENERATION TESTS (Target: [13, 11, 13, 11])

--- Test 1 ---
Theme   : शृंगार
Context : नायिका की सुंदरता का वर्णन

Result:
मानी हर प्राण मिले, | सब बनाया का बैठा || नायिका की सुंदरता का | नाय की स’ नाय
Matras      : [13, 13, 15, 9]
Matra Error : 6
Validation  : ❌ FAIL

--- Test 2 ---
Theme   : भक्ति
Context : ईश्वर के प्रति समर्पण

Result:
मानवा निर्मल करने में, | चार भर तक बिना || ईश्वर के प्रति समर्पण || बने आया साकुर कार
Matras      : [15, 10, 13, 14]
Matra Error : 6
Validation  : ❌ FAIL

--- Test 3 ---
Theme   : नीति
Context : सज्जन पुरुषों का स्वभाव

Result:
कोई जाने स्वप्न में, | तुम रहन से सज्जन || सुनाई के सजन मानि, | दुर्लभ कहत कोई 
Matras      : [13, 11, 13, 11]
Matra Error : 0
Validation  : ✅ PASS

--- Test 4 ---
Theme   : प्रकृति
Context : वर्षा ऋतु का सुंदर दृश्य

Result:
चित्र किन्तु समय का स्वप्न || कोई हित है वर्षा है, | देखो वर् के लिए का, | साधना है हरि दृश्य
Matras      : [14, 14, 13, 12]
Matra Error : 5
Validation  : ❌ FAIL


In [8]:
from transformers import AutoTokenizer, T5ForConditionalGeneration
import torch

# Test Stage 1 directly
tokenizer = AutoTokenizer.from_pretrained("KGan31/Doha-Gen-Stage2")
model = T5ForConditionalGeneration.from_pretrained("KGan31/Doha-Gen-Stage2")

# Test 1: Does it understand the prompt format at all?
inputs = tokenizer(
    "विषय: भक्ति | संदर्भ: ईश्वर के प्रति समर्पण | नियम: 13-11-13-11 मात्रा",
    return_tensors='pt'
)
out = model.generate(**inputs, num_beams=5, max_new_tokens=150)
print("Stage 1 output:", tokenizer.decode(out[0], skip_special_tokens=True))

# Test 2: Does it generate valid Hindi at all?
inputs2 = tokenizer("राम नाम", return_tensors='pt')
out2 = model.generate(**inputs2, num_beams=5, max_new_tokens=50)
print("Free generation:", tokenizer.decode(out2[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Stage 1 output: ईश्वर के प्रति समर्पण समर्पण के प्रति समर्पण के प्रति सम
Free generation: राम नाम नाम नाम नाम 
